In [1]:
#Source: https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/evaluate_rag_gen_ai_evaluation_service_sdk.ipynb

import vertexai
import inspect
import logging
import warnings

# General
from IPython.display import HTML, Markdown, display
import pandas as pd
import plotly.graph_objects as go

# Main
from vertexai.evaluation import EvalTask, MetricPromptTemplateExamples, PointwiseMetric

In [2]:
# Configuración del cliente de Vertex AI
PROJECT_ID = "dataton-2024-team-01-cofares"
LOCATION = "us-central1"
EXPERIMENT = "rag-eval-01"

vertexai.init(project=PROJECT_ID, location=LOCATION)

logging.getLogger("urllib3.connectionpool").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

In [3]:
# ----------------------Helper functions----------------------

def print_doc(function):
    print(f"{function.__name__}:\n{inspect.getdoc(function)}\n")


def display_eval_report(eval_result, metrics=None):
    """Display the evaluation results."""

    title, summary_metrics, report_df = eval_result
    metrics_df = pd.DataFrame.from_dict(summary_metrics, orient="index").T
    if metrics:
        metrics_df = metrics_df.filter(
            [
                metric
                for metric in metrics_df.columns
                if any(selected_metric in metric for selected_metric in metrics)
            ]
        )
        report_df = report_df.filter(
            [
                metric
                for metric in report_df.columns
                if any(selected_metric in metric for selected_metric in metrics)
            ]
        )

    # Display the title with Markdown for emphasis
    display(Markdown(f"## {title}"))

    # Display the metrics DataFrame
    display(Markdown("### Summary Metrics"))
    display(metrics_df)

    # Display the detailed report DataFrame
    display(Markdown("### Report Metrics"))
    display(report_df)


def display_explanations(df, metrics=None, n=1):
    style = "white-space: pre-wrap; width: 800px; overflow-x: auto;"
    df = df.sample(n=n)
    if metrics:
        df = df.filter(
            ["instruction", "context", "reference", "completed_prompt", "response"]
            + [
                metric
                for metric in df.columns
                if any(selected_metric in metric for selected_metric in metrics)
            ]
        )

    for index, row in df.iterrows():
        for col in df.columns:
            display(HTML(f"{col}: {row[col]}"))
        display(HTML(""))


def plot_radar_plot(eval_results, max_score=5, metrics=None):
    fig = go.Figure()

    for eval_result in eval_results:
        title, summary_metrics, report_df = eval_result

        if metrics:
            summary_metrics = {
                k: summary_metrics[k]
                for k, v in summary_metrics.items()
                if any(selected_metric in k for selected_metric in metrics)
            }

        fig.add_trace(
            go.Scatterpolar(
                r=list(summary_metrics.values()),
                theta=list(summary_metrics.keys()),
                fill="toself",
                name=title,
            )
        )

    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, max_score])), showlegend=True
    )

    fig.show()


def plot_bar_plot(eval_results, metrics=None):
    fig = go.Figure()
    data = []

    for eval_result in eval_results:
        title, summary_metrics, _ = eval_result
        if metrics:
            summary_metrics = {
                k: summary_metrics[k]
                for k, v in summary_metrics.items()
                if any(selected_metric in k for selected_metric in metrics)
            }

        data.append(
            go.Bar(
                x=list(summary_metrics.keys()),
                y=list(summary_metrics.values()),
                name=title,
            )
        )

    fig = go.Figure(data=data)

    # Change the bar mode
    fig.update_layout(barmode="group")
    fig.show()

In [12]:
# ----------------------Dataset----------------------

"""To evaluate the RAG generated answers,
the evaluation dataset is required to contain the following fields:

Prompt: The user supplied prompt consisting of the User Question and the RAG Retrieved Context
Response: The RAG Generated Answer

Your dataset must include a minimum of one evaluation example.
We recommend around 100 examples to ensure high-quality aggregated metrics
and statistically significant results."""

#El siguiente template a sido generado con Gemini a modo de ejemplo de implementación

import pandas as pd

# Ejemplos de preguntas de los usuarios
questions = [
    """Busco una crema para las estrías""",
    """Necesito un suplemento de vitamina D para personas mayores""",
    """¿Tienes algún producto para la caída del cabello?""",
    """Quiero una crema hidratante para piel sensible""",
    """¿Hay algún spray nasal para alergias?"""
]

# Contexto recuperado por el sistema RAG (simulación de descripciones de productos relevantes)
retrieved_contexts_by_bigquery = [
    """Nombre: Duplo Farline Crema De Manos Anti_Age, 2 x 50 ml, Descripción: Duplo Farline Crema De Manos Anti-Age ayuda a aclarar las manchas en las manos y contribuye a evitar su aparición. Esta es una crema de acción especial antiedad, cuyos componentes ayudan a difuminar las arrugas y manchas cutáneas: alteraciones de la piel que se presentan en las manos como consecuencia de la exposición solar extrema, así como a causa de la sequedad producida por agentes contaminantes. Al ser aplicada diariamente sobre las manos logra nutrir la piel en profundidad, creando una barrera contra elementos dañinos y rayos solares. Además, es una crema de fácila absorción.Usada regularmente, esta crema de manos ayuda a suavizar la piel y la protege de agentes externos como las radiaciones solares UV, los jabones abrasivos, la sequedad producida por el polvo, el aire frío en invierno y los cambios de temperatura. Entre otros ingredientes contiene aceite de oliva y pantenol, que ejercen una acción suavizante en las manos. El producto está indicado para un público de mediana edad en adelante, o que por cualquier motivo presente una piel envejecida o problemática en las manos. Se presenta en un formato duplo, que incluye dos tubos de esta crema antiedad Farline de 50ml cada uno., Modo de implementación: Te recomendamos aplicar la crema sobre las manos y masajear suavemente hasta su completa absorción.IndicacionesIndicada especialmente para las manos resecas, con manchas o en casos de exposición frecuente a agentes abrasivos.ContraindicacionesConservar en un lugar fresco y seco.No ingerir.Mantener alejado del alcance de los niños., Distancia: 0.35373178452692056 \
    Nombre: CREMA ACEITE ROSA MOSQU 50ML, Descripción: "CREMA DE ACEITE ROSA MOSQUETA. Presentado en un envase de 50 ml. Nutre, regenera, repara y protege la piel, dando una sensación inmediata de suavidad. Contribuye a reducir las arrugas. Mejora el estado de las cicatrices. Aporta a la piel la hidratación y emoliencia necesarias para evitar la sequedad cutánea. Protege de agresiones externas como las variaciones de humedad y temperatura ambientales. Favorece la rápida recuperación del equilibrio fisiológico de la piel.  Sin parabenos, no testado en animales.  Precauciones: Evitar contacto con los ojos y mucosas.", Modo de implementación: Aplicar dos veces al día, mañana y noche, después de limpiar y tonificar la piel. Repartir la crema por la cara, cuello y escote dando un suave masaje con las yemas de los dedos hasta su completa absorción., Distancia: 0.3748334345316666
    Nombre: CREMA REAFIRMANTE 500 ML, Descripción: CREMA REAFIRMANTE ESPECIAL FLACIDEZ 500 ML. Reafirmante concentrado unisex que hidrata y regenera piel, suavizando los signos de la flacidez cutánea y dotándole de un aspecto más liso y tonificado. Apto para todo tipo de pieles. Textura fluida y ligera y de fácil absorción, resulta ideal su uso para todo el cuerpo. No deja sensación grasa. No testado en animales., Modo de implementación: Aplicar preferentemente después del baño o ducha, mediante suaves masajes en todo el cuerpo., Distancia: 0.3855254374369854
    Nombre: BELLA AURORA SPLENDOR 60 CREMA DE NOCHE FORTIFICANTE ANTIEDAD 50 MILILITROS, Descripción: Tratamiento anti-edad desarrollado para las pieles más exigentes. Con activos anti-edad de última generación y extractos naturales., Modo de implementación: "Aplicar una pequeña cantidad sobre rostro, cuello y escote con la piel limpia o después del serum, realizando un suave masaje hasta su completa absorción. ", Distancia: 0.3977800061161346
    Nombre: Pack Eucerin Ph5 Bálsamo Nutritivo + Oleogel Ducha, 450 ml + 400 ml, Descripción: Pack Eucerin Ph5 Bálsamo Nutritivo + Oleogel Ducha es un pack con un contenido neto total de 850 ml de producto. En este pack viene oleogel de ducha, el cual está indicado para el cuidado de las pieles sensibles y secas. Este producto ayuda a prevenir la sequedad y descamación cutánea. También colabora con la preservación del manto ácido protector de la piel. Su fórmula ayuda a incrementar los lípidos de la superficie de la piel hasta un mínimo de 140 %. Viene en una presentación con 400 ml, los cuales son más que suficientes para observar resultados en la piel. El resultado del uso constante de este producto es una piel limpia y protegida frente a las agresiones externas.El otro producto del pack es un bálsamo para pieles sensibles. Es muy suave y efectivo para la cara y el cuerpo, ya que contribuye a la regeneración de las defensas naturales de la piel. Este bálsamo se ha desarrollado para ser aplicado sobre la piel corporal y facial sensible. La fórmula contiene ciertos ingredientes activos que estimulan la regeneración de la piel y fortalece la barrera protectora de la misma. También defiende la piel frente a la irritación, ayuda a restaurar los niveles de pH y repone las reservas de hidratantes de la propia piel hasta por 24 horas., Modo de implementación: Te recomendamos aplicar el oleogel durante la ducha y después de la misma el bálsamo.IndicacionesIndicado para lograr un cuidado e hidratación óptima de la piel del cuerpo y cara.ContraindicacionesNo utilizar en personas sensibles a sus componentes.No utilizar en niños menores de 2 años, en caso de ser necesaria su aplicación, consultar previamente al pediatra o farmacéutico.Conservar en un lugar fresco y seco.No ingerir.Evitar el contacto con los ojos.Mantener alejado de fuentes de luz y calor., Distancia: 0.39962154137837846
    Nombre: Pack Duplo Farline Activity Vaselina Anti_rozaduras, 2 x 60 ml, Descripción: Farline Vaselina Anti-rozaduras está formulada con Aceite de Oliva y contiene vitaminas E, B5 y B3, además de aceites esenciales de lavanda, pino y romero.Estos ingredientes permiten proteger y regenerar la piel, creando una película protectora que la aísla de la superficie con la que está en contacto y que provoca la rozadura., Modo de implementación: Te recomendamos aplicar la vaselina directamente sobre la superficie del cuerpo que quiera protegerse y/o suavizarse.IndicacionesIndicado como ayuda para evitar las rozaduras y la formación de ampollas debidas a la fricción.ContraindicacionesUso tópico.No aplicar sobre la piel lesionada (ampollas o heridas abiertas,...).Mantener fuera del alcance de los niños.No ingerir.No apto para su uso en mucosas.Si experimenta reacciones alérgicas a alguno de los componentes, interrumpa inmediatamente su uso., Distancia: 0.40241873228252545
    Nombre: Pack Eucerin Urea Repair Plus Bálsamo Nutritivo + Gel Baño, 450 ml + 400 ml, Descripción: Pack Eucerin Urea Repair Plus Bálsamo Nutritivo + Gel Baño con un poderoso efecto si se utiliza en conjunto. El bálsamo nutritivo tiene una acción aproximada de 48 horas. Se encarga de revitalizar y suavizar la piel. Una vez aplicado este producto, no permite que regresen los síntomas de la piel seca y deshidratada. Esta crema se puede aplicar en todo el cuerpo, teniendo un efecto similar en cada zona en la que se aplica. Este bálsamo permite fortalecer la capa natural de la piel, al tiempo que ofrece proteínas y demás nutrientes. Puede ser usado de forma complementaria en personas que requieran atención médica para su piel. No está contraindicado en casos de psoriasis, diabetes y piel madura. Por su parte, el gel de baño es suave y reparador para pieles secas y muy secas. Tiene un efecto considerablemente rápido sobre la piel afectada. Se utiliza para mejorar los síntomas de la resequedad corporal. Aporta minerales y demás nutrientes al cuerpo. Es completamente de uso externo, por lo que no se recomienda colocar en otras cavidades del cuerpo. Este gel de baño enriquecido con urea, también contiene lactato, una sustancia responsable de la hidratación corporal. Estos productos están testados dermatológicamente y no representan un riesgo para la salud. Son hipoalergénicos., Modo de implementación: Te recomendamos aplicar sobre la piel, masajeando suavemente hasta hacer espuma. Aclarar con abundante agua.IndicacionesIndicado para limpiar la piel de agentes contaminantes que están presentes en el medio ambiente. ContraindicacionesNo utilizar en personas sensibles a sus componentes.No utilizar en niños menores de 2 años, en caso de ser necesaria su aplicación, consultar previamente al pediatra o farmacéutico.Usar con precaución en pieles sensibles.Conservar en un lugar fresco y seco.Mantener alejado de fuentes de luz y calor.Mantener alejado del alcance de los niños., Distancia: 0.4046472548438982
    Nombre: Pack Uresim Serum Ác.Hialurónico 30 ml + Crema Nutri+ 50 ml, Descripción: Pack Uresim Serum Ác.Hialurónico y Crema Nutri+El serum antiedad de Uresim con una alta concentración de ácido hialurónico, constituye un tratamiento integral para ayudar a la hidratación de la piel y prevenir el envejecimiento prematuro. Contribuye a prevenir las arrugas y líneas de expresión.Además ayuda a aportar tersura y suavidad a la piel. La Crema Nutri+ es nutritiva y está indicada para pieles secas y maduras. Contiene alta concentración de aceites vegetales, como el aceite de Rosa Mosqueta, aceite de Macadamia, manteca de Karité y aceite de Soja. Esta formulación ayuda a aportar nutrientes esenciales como ácidos grasos, isoflavonas de soja y  vitaminas  A, C y E a la piel. El ácido hialurónico y el extracto de caviar contribuye a la microcirculación t la oxigenación de la piel., Modo de implementación: Te recomendamos que la Uresim Pure Hyaluronic Acid Serum lo apliques por la mañana y/o noche sobre piel limpia y seca con un suave masaje hasta su total absorción. Puede empelarse como base de día o como reparador nocturno.La Crema Nutri+ Uresim la puedes aplicar día y/o noche.IndicacionesIndicado para adultos.ContraindicacionesEste tratamiento está especialmente indicado para pieles secas, deshidratadas y con aspecto cansado y apagado., Distancia: 0.40913467599995934
    Nombre: MELASES CYSTEAMINE CR GEL 50ML, Descripción: Sesderma Melases Cysteamine es una crema gel para piel con hiperpigmentación, tendencia a melasma o signos de fotoenvejecimiento. Apta para todo tipo de pieles y recomendada para fototipos altos. Fototipo es el término que se utiliza para describir la respuesta de la piel a la exposición solar., Modo de implementación: Con el rostro limpio y seco, aplica 3 o 4 pulsaciones en los dedos y masajea rostro y cuello hasta su completa absorción. También puede usarse en axilas, codos, piernas, manos y brazos. Evita el contacto con los ojos. Se recomienda uso diario, mañana y noche., Distancia: 0.4112510990089042
    Nombre: SCHUSSLER CR FAC ARBOL TE 75ML, Descripción: Enriquecida con aceite de árbol de té, nuestra crema facial natural combate eficazmente el acné, las espinillas y el llamado maskne que es un tipo especial de acné, que aparece como resultado del uso de mascarillas., Modo de implementación: Use esta crema facial como parte de su rutina diaria de cuidado de la piel: después de limpiar y secar la piel con palmaditas, aplique 1-2 cantidades del tamaño de un guisante en su cara o en el lugar de destino. Masajee suavemente la piel. La crema se absorbe rápidamente sin dejar ningún residuo graso., Distancia: 0.41358794798064347""",
    """Nombre: VITAMINAS D3&K2 60CAPS, Descripción: -, Modo de implementación: -, Distancia: 0.34517607999169775
    Nombre: BIG VITAMINA D3 + K2 -120 VEGICAPS, Descripción: Sinergia de dos nutrientes clave para la salud ósea y cardiovascular. Apto para vegetarianos., Modo de implementación: Tomar una VegCap al día con la comida o con un vaso de agua., Distancia: 0.3502689031446813
    Nombre: Duplo Aquilea Colágeno + Magnesio, 2 x 375 g, Descripción: Duplo Aquilea Colágeno + Magnesio destaca por su contenido en:Magnesio, que contribuye al funcionamiento normal de los músculos, al mantenimiento de los huesos en condiciones normales y al metabolismo energético normal.Vitamina C, que contribuye a la formación normal de colágeno para el funcionamiento normal de los huesos y de os cartílagos.Sabor a limón., Modo de implementación: Te recomendamos tomar 1 cucharada de 12,5 g (un cacito) al día disuelto en un vaso de agua.IndicacionesIndicado para adultos.ContraindicacionesMantener fuera del alcance de los niños.Conservar en un lugar fresco y seco.No superar la dosis diaria recomendada.Los complementos alimenticios no deben sustituir una dieta variada y saludable.No consumir una vez pasada la fecha de caducidad que aparece en el envase., Distancia: 0.3762568670490928
    Nombre: K2 + D3+ SILICIO 60 COMP, Descripción: K2 + D3 + Silicio de Natysal es un complemento alimenticio con un alto contenido de Vitamina K2 y D3, con Acido ortosilícico estabilizado con colina, Modo de implementación: Se recomienda tomar 1 comprimido al día, ingiriéndolo o disolviéndolo en la boca, Distancia: 0.38284188623097903
    Nombre: LIPOSOMAL MULTI WOMENS 60 CAP, Descripción: Liposomal Multivitamin Women’s es una fórmula multivitamínica especialmente diseñada para mujeres, que reúne una poderosa combinación de nutrientes para impulsar la salud general y el bienestar. Esta completa mezcla de vitaminas, minerales y extractos vegetales no solo promueve la vitalidad femenina, sino que también respalda el cuidado de la belleza., Modo de implementación: Tome 2 cápsulas de Liposomal Multivitamin Women’s Solaray acompañadas de un vaso de agua, preferiblemente con una comida para una absorción óptima. Este sencillo hábito diario le brindará el respaldo necesario para mantenerse en su mejor forma a medida que abraza cada día con energía y vitalidad., Distancia: 0.3835481549806943
    Nombre: LIPOSOMAL MULTIVIT 250 MILILITROS, Descripción: Complemento alimenticio LIPOSOMAL MULTIVIT 250 ml. Producto apto para vegetarianos. No contiene maíz, levadura o arroz. Sin gluten, Modo de implementación: Tomar una cucharilla (5 ml) al día, con o sin comida. Agitar antes de tomar. Puede tomarse directamente o mezclado con agua o zumo. Conservar en lugar fresco y seco. Una vez abierto guardar en el frigorífico. Los complementos alimenticios no deben utilizarse como sustitutos de una dieta variada y equilibrada. No superar la dosis diaria expresamente recomendada. Mantener fuera del alcance de los niños más pequeños. Un consumo excesivo puede producir efectos laxantes., Distancia: 0.3857758295417424
    Nombre: C1000 ACCIÓN RETARDADA100+20CO, Descripción: La vitamina C es una vitamina esencial para el ser humano que contribuye al funcionamiento normal del sistema inmunitario y a la formación de colágeno para el funcionamiento normal de los vasos sanguíneos, huesos, piel, encías y dientes También ayuda a disminuir el cansancio, la fatiga y mejora la absorción de hierro, Modo de implementación: Se recomienda tomar 1 comprimido por día preferiblemente con la comida, Distancia: 0.3893107112786208
    Nombre: VIGOR SOL ACTIF PLUS PERLAS, Descripción: Complemento alimenticio con aceite de onagra y vitaminas (A,C,B2,B3,B8) y minerales como el zinc y el cobre, que contribuyen al mantenimiento de la piel en condiciones normales., Modo de implementación: Tomar 1 perla, a cualquier hora del día., Distancia: 0.39762914847604014
    Nombre: DEKORO SPRAY VIT D3+K2 20ML, Descripción: -, Modo de implementación: -, Distancia: 0.40954387836881345
    Nombre: PEPTIVIS LIMON 20 SOBRES, Descripción: Peptivis® es un complemento alimenticio a base de péptidos bioactivos de colágeno hidrolizado, HMB, Leucina y Vitamina D3. Ayuda a aumentar la masa muscular , mejorar la fuerza muscular , reducir la pérdida de capacidad motora y prevenir la pérdida de masa muscular y la sarcopenia., Modo de implementación: 2 sobres al día. Disolver el contenido del sobre en 150-200 ml de agua y mezclar bien., Distancia: 0.4125694536357126""",
    """Nombre: CURLY METHOD PACK, Descripción: CURLY METHOD PACK. Este pack contiene:  - Champú Final Wash: se utilizará las primeras 5 veces que apliques la rutina. Este champú es el único de la línea que contiene sulfatos, estos son necesarios para eliminar todos los residuos que se han ido depositando en nuestro cabello. Después de cada lavado de Final Wash, continuaremos con el paso 2, 3 y 4 de la rutina, para ver los efectos desde el primer día.  Una vez finalizadas las 5 primeras rutinas, sustituiremos el Final Wash, por el Champú Low-Poo, que será el champú definitivo para el resto de lavados.  - Mascarilla Co-wash: Hidrata, repara y desenreda el cabello. Ayuda a controlar el encrespamiento y a reducir la sequedad dejando los rizos suaves y brillantes.  - Crema de peinado Leave-in: Sin aclarado. Repara el cabello y elimina el efecto frizz. Ayuda a definir los rizos sin apelmazar.  - Activador de rizos Styling: Gel de definición que da forma, resalta y revitaliza los rizos con un aspecto natural. Fijación suave sin apelmazar., Modo de implementación: Aplicar sobre el cabello mojado el champú Final Wash (se utilizará las primeras 5 veces que apliques la rutina. Una vez finalizadas las 5 primeras rutinas, sustituiremos el Final Wash, por el Champú Low-Poo). Enjabonar bien y masajear el todo el cuero cabelludo. Aclarar con abundante agua.  Continuar con la Mascarilla Curly para una hidratación profunda.  Para evitar el encrespamiento y que nuestros rizos tengan un aspecto revitalizado y definido, aplicar la crema reparadora sin aclarado.  Tras el uso de la crema de peinado y con el cabello aún húmedo, aplicar el gel repartiéndolo y dándole forma al rizo con las manos. Por último, secar al aire o usar difusor a baja temperatura, tocando el cabello lo menos posible, para evitar frizz o encrespamiento., Distancia: 0.3825830042098063
    Nombre: SOLARAY PIEL CABELLO Y UÑAS 60 CAPSULAS, Descripción: Solaray Piel, Cabello y Uñas 60 Cápsulas Vegetales. Solaray Pelo, Piel y Uñas te ofrece la solución perfecta. Esta fórmula única contiene una mezcla de aminoácidos, minerales, vitaminas y antioxidantes que proporcionan los nutrientes esenciales a tu cabello, uñas y piel., Modo de implementación: Tomar una Vegcap al día, con la comida o con un vaso de agua., Distancia: 0.3916230192519107
    Nombre: CAPILARE ANTICAIDA HOMBRE Y MUJER 120 CAPSULAS, Descripción: -, Modo de implementación: -, Distancia: 0.414020754328834
    Nombre: FARLINE DUPLO CHAMPÚ FORTIFICANTE, Descripción: -, Modo de implementación: -, Distancia: 0.4180371054773335
    Nombre: FARLINE DUPLO CHAMPU FRECUENCIA 2 X 500 ML, Descripción: -, Modo de implementación: -, Distancia: 0.43049833646282376
    Nombre: CEPILLO RAQUETA BAMWOOD 03117, Descripción: -, Modo de implementación: -, Distancia: 0.4315995198980762
    Nombre: NUK Cepillo Extrasuave, 1 Unidad, Descripción: Cepillo para peinar a los más pequeños, fabricado con cerdas de pelo 100% natural. Son extrasuaves, lo que permite un masaje del cuero cabelludo delicado. Su mango es antideslizante. Es perfecto para bebés recién nacidos.El color dependerá del stock disponible en la farmacia., Modo de implementación: Masajear suavemente la cabeza del bebé.IndicacionesPara cepillar y masajear el cuero cabelludo de los bebés.ContraindicacionesMantener alejado de fuentes de calor. No utilizar si alguno de sus componentes está dañado o en malas condiciones, Distancia: 0.43408808472099814
    Nombre: CREMA REAFIRMANTE 500 ML, Descripción: CREMA REAFIRMANTE ESPECIAL FLACIDEZ 500 ML. Reafirmante concentrado unisex que hidrata y regenera piel, suavizando los signos de la flacidez cutánea y dotándole de un aspecto más liso y tonificado. Apto para todo tipo de pieles. Textura fluida y ligera y de fácil absorción, resulta ideal su uso para todo el cuerpo. No deja sensación grasa. No testado en animales., Modo de implementación: Aplicar preferentemente después del baño o ducha, mediante suaves masajes en todo el cuerpo., Distancia: 0.44209098220027265
    Nombre: Pack Farline con Árbol de Té Champú+Spray, 1 Ud, Descripción: Pack Farline con Árbol de Té Champú+Spray contiene los esenciales para el cuidado del cabello de los más pequeños con un agradable perfume a fresa. El spray ayuda a facilitar el peinado y el champú es de uso diario., Modo de implementación: Utilizar el champú sobre el cabello, realizando un masaje sobre el cuero cabelludo hasta que aparezca espuma. Aclarar con abundante agua. Aplicar varias pulverizaciones del spray sobre el cabello limpio y húmedo, cepillando con un peine para desenredar y extender el producto.IndicacionesIndicado para el cuidado del cabello de los niños.ContraindicacionesEvitar el contacto con los ojos.No ingerir.Mantener fuera del alcance de los niños. Conservar en un lugar fresco y seco., Distancia: 0.4421944964566139
    Nombre: CREMA ACEITE ROSA MOSQU 50ML, Descripción: "CREMA DE ACEITE ROSA MOSQUETA. Presentado en un envase de 50 ml. Nutre, regenera, repara y protege la piel, dando una sensación inmediata de suavidad. Contribuye a reducir las arrugas. Mejora el estado de las cicatrices. Aporta a la piel la hidratación y emoliencia necesarias para evitar la sequedad cutánea. Protege de agresiones externas como las variaciones de humedad y temperatura ambientales. Favorece la rápida recuperación del equilibrio fisiológico de la piel.  Sin parabenos, no testado en animales.  Precauciones: Evitar contacto con los ojos y mucosas.", Modo de implementación: Aplicar dos veces al día, mañana y noche, después de limpiar y tonificar la piel. Repartir la crema por la cara, cuello y escote dando un suave masaje con las yemas de los dedos hasta su completa absorción., Distancia: 0.44775944973437""",
    """Nombre: CREMA REAFIRMANTE 500 ML, Descripción: CREMA REAFIRMANTE ESPECIAL FLACIDEZ 500 ML. Reafirmante concentrado unisex que hidrata y regenera piel, suavizando los signos de la flacidez cutánea y dotándole de un aspecto más liso y tonificado. Apto para todo tipo de pieles. Textura fluida y ligera y de fácil absorción, resulta ideal su uso para todo el cuerpo. No deja sensación grasa. No testado en animales., Modo de implementación: Aplicar preferentemente después del baño o ducha, mediante suaves masajes en todo el cuerpo., Distancia: 0.33572734167627827
    Nombre: Pack Eucerin Ph5 Bálsamo Nutritivo + Oleogel Ducha, 450 ml + 400 ml, Descripción: Pack Eucerin Ph5 Bálsamo Nutritivo + Oleogel Ducha es un pack con un contenido neto total de 850 ml de producto. En este pack viene oleogel de ducha, el cual está indicado para el cuidado de las pieles sensibles y secas. Este producto ayuda a prevenir la sequedad y descamación cutánea. También colabora con la preservación del manto ácido protector de la piel. Su fórmula ayuda a incrementar los lípidos de la superficie de la piel hasta un mínimo de 140 %. Viene en una presentación con 400 ml, los cuales son más que suficientes para observar resultados en la piel. El resultado del uso constante de este producto es una piel limpia y protegida frente a las agresiones externas.El otro producto del pack es un bálsamo para pieles sensibles. Es muy suave y efectivo para la cara y el cuerpo, ya que contribuye a la regeneración de las defensas naturales de la piel. Este bálsamo se ha desarrollado para ser aplicado sobre la piel corporal y facial sensible. La fórmula contiene ciertos ingredientes activos que estimulan la regeneración de la piel y fortalece la barrera protectora de la misma. También defiende la piel frente a la irritación, ayuda a restaurar los niveles de pH y repone las reservas de hidratantes de la propia piel hasta por 24 horas., Modo de implementación: Te recomendamos aplicar el oleogel durante la ducha y después de la misma el bálsamo.IndicacionesIndicado para lograr un cuidado e hidratación óptima de la piel del cuerpo y cara.ContraindicacionesNo utilizar en personas sensibles a sus componentes.No utilizar en niños menores de 2 años, en caso de ser necesaria su aplicación, consultar previamente al pediatra o farmacéutico.Conservar en un lugar fresco y seco.No ingerir.Evitar el contacto con los ojos.Mantener alejado de fuentes de luz y calor., Distancia: 0.3503244068166941
    Nombre: Pack Eucerin Urea Repair Plus Bálsamo Nutritivo + Gel Baño, 450 ml + 400 ml, Descripción: Pack Eucerin Urea Repair Plus Bálsamo Nutritivo + Gel Baño con un poderoso efecto si se utiliza en conjunto. El bálsamo nutritivo tiene una acción aproximada de 48 horas. Se encarga de revitalizar y suavizar la piel. Una vez aplicado este producto, no permite que regresen los síntomas de la piel seca y deshidratada. Esta crema se puede aplicar en todo el cuerpo, teniendo un efecto similar en cada zona en la que se aplica. Este bálsamo permite fortalecer la capa natural de la piel, al tiempo que ofrece proteínas y demás nutrientes. Puede ser usado de forma complementaria en personas que requieran atención médica para su piel. No está contraindicado en casos de psoriasis, diabetes y piel madura. Por su parte, el gel de baño es suave y reparador para pieles secas y muy secas. Tiene un efecto considerablemente rápido sobre la piel afectada. Se utiliza para mejorar los síntomas de la resequedad corporal. Aporta minerales y demás nutrientes al cuerpo. Es completamente de uso externo, por lo que no se recomienda colocar en otras cavidades del cuerpo. Este gel de baño enriquecido con urea, también contiene lactato, una sustancia responsable de la hidratación corporal. Estos productos están testados dermatológicamente y no representan un riesgo para la salud. Son hipoalergénicos., Modo de implementación: Te recomendamos aplicar sobre la piel, masajeando suavemente hasta hacer espuma. Aclarar con abundante agua.IndicacionesIndicado para limpiar la piel de agentes contaminantes que están presentes en el medio ambiente. ContraindicacionesNo utilizar en personas sensibles a sus componentes.No utilizar en niños menores de 2 años, en caso de ser necesaria su aplicación, consultar previamente al pediatra o farmacéutico.Usar con precaución en pieles sensibles.Conservar en un lugar fresco y seco.Mantener alejado de fuentes de luz y calor.Mantener alejado del alcance de los niños., Distancia: 0.35946430888261427
    Nombre: SCHUSSLER AC FAC AVENA 25ML, Descripción: Aceite facial intensamente nutritivo y calmante para una piel aterciopelada, suave y resplandeciente, recomendado para todo tipo de piel. Con una fórmula repleta de valiosos aceites vegetales, escualano y dos tipos de sales tisulares de Schüssler para hidratar y suavizar la piel de forma eficaz y protegerla del daño externo. Especialmente recomendado para pieles sensibles con tendencia a la rosácea. Incorpóralo a tu rutina diaria de cuidado de la piel. Con una fórmula ligera que no dejará residuos grasos en la piel., Modo de implementación: Masajee suavemente unas gotas en la piel después de celarla completamente y antes de pasar al siguiente paso de su rutina, por ejemplo, aplicar una crema facial hidratante y nutritiva., Distancia: 0.36627977226420894
    Nombre: CREMA ACEITE ROSA MOSQU 50ML, Descripción: "CREMA DE ACEITE ROSA MOSQUETA. Presentado en un envase de 50 ml. Nutre, regenera, repara y protege la piel, dando una sensación inmediata de suavidad. Contribuye a reducir las arrugas. Mejora el estado de las cicatrices. Aporta a la piel la hidratación y emoliencia necesarias para evitar la sequedad cutánea. Protege de agresiones externas como las variaciones de humedad y temperatura ambientales. Favorece la rápida recuperación del equilibrio fisiológico de la piel.  Sin parabenos, no testado en animales.  Precauciones: Evitar contacto con los ojos y mucosas.", Modo de implementación: Aplicar dos veces al día, mañana y noche, después de limpiar y tonificar la piel. Repartir la crema por la cara, cuello y escote dando un suave masaje con las yemas de los dedos hasta su completa absorción., Distancia: 0.37240970352958747
    Nombre: Pack Uresim Serum Ác.Hialurónico 30 ml + Crema Nutri+ 50 ml, Descripción: Pack Uresim Serum Ác.Hialurónico y Crema Nutri+El serum antiedad de Uresim con una alta concentración de ácido hialurónico, constituye un tratamiento integral para ayudar a la hidratación de la piel y prevenir el envejecimiento prematuro. Contribuye a prevenir las arrugas y líneas de expresión.Además ayuda a aportar tersura y suavidad a la piel. La Crema Nutri+ es nutritiva y está indicada para pieles secas y maduras. Contiene alta concentración de aceites vegetales, como el aceite de Rosa Mosqueta, aceite de Macadamia, manteca de Karité y aceite de Soja. Esta formulación ayuda a aportar nutrientes esenciales como ácidos grasos, isoflavonas de soja y  vitaminas  A, C y E a la piel. El ácido hialurónico y el extracto de caviar contribuye a la microcirculación t la oxigenación de la piel., Modo de implementación: Te recomendamos que la Uresim Pure Hyaluronic Acid Serum lo apliques por la mañana y/o noche sobre piel limpia y seca con un suave masaje hasta su total absorción. Puede empelarse como base de día o como reparador nocturno.La Crema Nutri+ Uresim la puedes aplicar día y/o noche.IndicacionesIndicado para adultos.ContraindicacionesEste tratamiento está especialmente indicado para pieles secas, deshidratadas y con aspecto cansado y apagado., Distancia: 0.3749166286648212
    Nombre: SCHUSSLER SERUM AC HIALUR 30ML, Descripción: Suero facial de ácido hialurónico con efecto antiarrugas y tensor de la piel. Se puede usar en la cara, el cuello y el escote para que la piel se sienta suave e hidratada. Sus efectos hidratantes profundos y duraderos se sienten inmediatamente después del primer uso. Fórmula sin aceite para igualar y suavizar la textura de la piel sin dejarla grasosa. Recomendado para pieles deshidratadas., Modo de implementación: -, Distancia: 0.38652055716603984
    Nombre: SCHUSSLER AC FAC BAKUCHIO 25ML, Descripción: Aceite facial suavizante con bakuchiol y sales tisulares de Schüssler para una piel hidratada, más firme y un cutis radiante. El bakuchiol, la suave alternativa vegana al retinol, ayuda a reducir los signos visibles de las arrugas y la decoloración de la pigmentación, estimula la regeneración de la piel y la producción de colágeno con sus fuertes propiedades antioxidantes, todo esto sin irritar la piel., Modo de implementación: Masajee suavemente unas gotas en la piel después de celarla completamente y antes de pasar al siguiente paso de su rutina. Úselo localmente y apunte a puntos y áreas específicas de su rostro después de aplicar la crema hidratante., Distancia: 0.39044636463488924
    Nombre: BELLA AURORA SPLENDOR 60 CREMA DE NOCHE FORTIFICANTE ANTIEDAD 50 MILILITROS, Descripción: Tratamiento anti-edad desarrollado para las pieles más exigentes. Con activos anti-edad de última generación y extractos naturales., Modo de implementación: "Aplicar una pequeña cantidad sobre rostro, cuello y escote con la piel limpia o después del serum, realizando un suave masaje hasta su completa absorción. ", Distancia: 0.3949296082580378
    Nombre: Duplo Farline Crema De Manos Anti_Age, 2 x 50 ml, Descripción: Duplo Farline Crema De Manos Anti-Age ayuda a aclarar las manchas en las manos y contribuye a evitar su aparición. Esta es una crema de acción especial antiedad, cuyos componentes ayudan a difuminar las arrugas y manchas cutáneas: alteraciones de la piel que se presentan en las manos como consecuencia de la exposición solar extrema, así como a causa de la sequedad producida por agentes contaminantes. Al ser aplicada diariamente sobre las manos logra nutrir la piel en profundidad, creando una barrera contra elementos dañinos y rayos solares. Además, es una crema de fácila absorción.Usada regularmente, esta crema de manos ayuda a suavizar la piel y la protege de agentes externos como las radiaciones solares UV, los jabones abrasivos, la sequedad producida por el polvo, el aire frío en invierno y los cambios de temperatura. Entre otros ingredientes contiene aceite de oliva y pantenol, que ejercen una acción suavizante en las manos. El producto está indicado para un público de mediana edad en adelante, o que por cualquier motivo presente una piel envejecida o problemática en las manos. Se presenta en un formato duplo, que incluye dos tubos de esta crema antiedad Farline de 50ml cada uno., Modo de implementación: Te recomendamos aplicar la crema sobre las manos y masajear suavemente hasta su completa absorción.IndicacionesIndicada especialmente para las manos resecas, con manchas o en casos de exposición frecuente a agentes abrasivos.ContraindicacionesConservar en un lugar fresco y seco.No ingerir.Mantener alejado del alcance de los niños., Distancia: 0.3977731362052894""",
    """Nombre: Pack Bexident Fresh Breath Colutorio + Spray, Descripción: La gama Fresh Breath de Bexident ayuda a combatir el mal aliento neutralizándolo y combatiendo los sulfuros volátiles que lo causan. Además, alivia la sequedad bucal y tiene propiedades antisépticas., Modo de implementación: Se recomienda usar el colutorio 3 veces al día, después del cepillado y durante, al menos, 30 segundos. Se recomienda pulverizar el spray 3 o 4 veces en la boca y no enjuagar.IndicacionesIndicado para conseguir una buena higiene bucal y un aliento fresco y sano.ContraindicacionesEvitar el contacto con los ojos. No ingerir. Mantener fuera del alcance de los niños., Distancia: 0.44051285926191364
    Nombre: FARLINE DUPLO FRIMAR BABY AGUA DE MAR + GASA BEBÉ REGALO 2 x 120 MILILITROS, Descripción: -, Modo de implementación: -, Distancia: 0.4449157475821408
    Nombre: Bach Rescue Spray, 20 ml, Descripción: Este spray está formulado a base de CherryPlum, Clematis, Impatiens, Rock Rose y Starof Bethelem. Esta combinación resulta efectiva para momentos de crisis nerviosas, estados de shock y angustia profunda. Ayuda también a encontrar la calma ante situaciones que van a suponer un estado de ansiedad o estrés., Modo de implementación: Pulverizar 2 veces sobre la lengua.IndicacionesIndicado para personas que necesitan alivio ante momentos de ansiedad o estrés.ContraindicacionesMantener fuera del alcance de los niños. Conservar en un lugar fresco y seco. Evitar el contacto con los ojos., Distancia: 0.45975424701834544
    Nombre: SCHUSSLER HIALUR SPY FAC 100ML, Descripción: Se recomienda para todo tipo de piel, pero puede ser un remedio único para la piel deshidratada, ya que se enfoca en la opacidad y la deshidratación: promete dejar la piel tersa, radiante y rejuvenecida. Úselo por la mañana y por la tarde después de limpiar la pie, Modo de implementación: Úselo en la mañana o en la tarde: después de limpiar su piel, espere 1-2 minutos para absorber la bruma facial y continúe con el siguiente paso en su ritual de cuidado de la piel: aplique un serum o su crema de día o de noche como de costumbre., Distancia: 0.4631758571867861
    Nombre: APOSAN ESENCIA HOJAS DE HIGUERA 10 MILILITROS, Descripción: -, Modo de implementación: -, Distancia: 0.4671607886721877
    Nombre: FINIQUITO BIO SPRAY 125 ML, Descripción: -, Modo de implementación: -, Distancia: 0.4718623511749558
    Nombre: Duplo Farline Desodorante Spray Extra_Dry, 2 Uds, Descripción: Farline Desodorante Spray Extra-Dry ayuda a controlar la sudoración excesiva, absorbiendo el exceso de humedad y aportando una máxima protección.Sin alcohol.Testado bajo control dermatológico., Modo de implementación: Te recomendamos aplicar Farline Desodorante Spray Extra-Dry manteniendo el envase en posición vertical y vaporizar a una distancia de 15 cm de la axila. Dejar secar completamente antes de vestir.IndicacionesIndicado para adultos.ContraindicacionesMantener fuera del alcance de los niños.Conservar en un lugar fresco y seco.Evitar el contacto con los ojos y la boca.No ingerir., Distancia: 0.472670669531567
    Nombre: Duplo Farline Desodorante para Hombre en Spray, 2 x 150 ml, Descripción: duplo farline desodorante para hombre en spray, ayuda al control de la sudoración excesiva, absorbiendo el exceso de humedad y ofreciendo una alta protección durante 48 horas. contiene alantoína y vitamina e.cuenta con una fórmula anti_manchas para la ropa. no contiene alcohol. cuida de la piel del hombre, incluso de la más sensible. está testado bajo control dermatológico., Modo de implementación: te recomendamos utilizar el desodorante tras la ducha, sobre la piel limpia y seca, pulverizando durante dos segundos a unos 15 cm de la axila.Indicacionesindicado para la higiene personal del hombre. Contraindicacionesmantener alejado de fuentes de calor.conservar en un lugar fresco y seco. no ingerir.evitar el contacto con los ojos y mucosas.mantener alejado del alcance de los niños., Distancia: 0.47551133082408104
    Nombre: SCHUSSLER NIACIN SPY FAC 100ML, Descripción: Nuestra bruma facial de niacinamida contiene la sal de tejido de Schüssler, Natrium phosphoricum, que desempeña un papel en la prevención del aumento de la acidez en el cuerpo. Si el equilibrio ácido-base de nuestro cuerpo se altera, nos deprimimos y agotamos mientras nuestra piel sufre de acné y puntos negros y se vuelve grasosa o áspera. Natrium phosphoricum puede ayudar a combatir el acné y los puntos negros causados ​​por el exceso de sebo al restaurar el equilibrio saludable de la producción de sebo., Modo de implementación: Úselo por la mañana y por la tarde después de la limpieza para preparar la piel para absorber ingredientes activos adicionales., Distancia: 0.47643730410977914
    Nombre: Duplo Farline Desodorante Spray Sensible, 2 Uds, Descripción: Farline Desodorante Spray Sensible no contiene sales de aluminio y aporta hasta 24 horas de protección. Apto para pieles sensibles.Sin alcohol.Testado dermatologicamente., Modo de implementación: Te recomendamos aplicar Duplo Farline Desodorante Spray Sensible manteniendo el envase en posición vertical y vaporizar a una distancia de 15 cm de la axila.IndicacionesIndicado para adultos.ContraindicacionesMantener fuera del alcance de los niños.Conservar en un lugar fresco y seco.Evitar el contacto con los ojos y la boca.No ingerir., Distancia: 0.47837449336449844"""
]

generated_answers_by_rerank = [
    """Nombre: CREMA ACEITE ROSA MOSQU 50ML, Descripción: "CREMA DE ACEITE ROSA MOSQUETA. Presentado en un envase de 50 ml. Nutre, regenera, repara y protege la piel, dando una sensación inmediata de suavidad. Contribuye a reducir las arrugas. Mejora el estado de las cicatrices. Aporta a la piel la hidratación y emoliencia necesarias para evitar la sequedad cutánea. Protege de agresiones externas como las variaciones de humedad y temperatura ambientales. Favorece la rápida recuperación del equilibrio fisiológico de la piel.  Sin parabenos, no testado en animales.  Precauciones: Evitar contacto con los ojos y mucosas.", Modo de implementación: Aplicar dos veces al día, mañana y noche, después de limpiar y tonificar la piel. Repartir la crema por la cara, cuello y escote dando un suave masaje con las yemas de los dedos hasta su completa absorción., Distancia: 0.3748334345316666
    Nombre: Pack Uresim Serum Ác.Hialurónico 30 ml + Crema Nutri+ 50 ml, Descripción: Pack Uresim Serum Ác.Hialurónico y Crema Nutri+El serum antiedad de Uresim con una alta concentración de ácido hialurónico, constituye un tratamiento integral para ayudar a la hidratación de la piel y prevenir el envejecimiento prematuro. Contribuye a prevenir las arrugas y líneas de expresión.Además ayuda a aportar tersura y suavidad a la piel. La Crema Nutri+ es nutritiva y está indicada para pieles secas y maduras. Contiene alta concentración de aceites vegetales, como el aceite de Rosa Mosqueta, aceite de Macadamia, manteca de Karité y aceite de Soja. Esta formulación ayuda a aportar nutrientes esenciales como ácidos grasos, isoflavonas de soja y  vitaminas  A, C y E a la piel. El ácido hialurónico y el extracto de caviar contribuye a la microcirculación t la oxigenación de la piel., Modo de implementación: Te recomendamos que la Uresim Pure Hyaluronic Acid Serum lo apliques por la mañana y/o noche sobre piel limpia y seca con un suave masaje hasta su total absorción. Puede empelarse como base de día o como reparador nocturno.La Crema Nutri+ Uresim la puedes aplicar día y/o noche.IndicacionesIndicado para adultos.ContraindicacionesEste tratamiento está especialmente indicado para pieles secas, deshidratadas y con aspecto cansado y apagado., Distancia: 0.40913467599995934
    Nombre: Pack Eucerin Urea Repair Plus Bálsamo Nutritivo + Gel Baño, 450 ml + 400 ml, Descripción: Pack Eucerin Urea Repair Plus Bálsamo Nutritivo + Gel Baño con un poderoso efecto si se utiliza en conjunto. El bálsamo nutritivo tiene una acción aproximada de 48 horas. Se encarga de revitalizar y suavizar la piel. Una vez aplicado este producto, no permite que regresen los síntomas de la piel seca y deshidratada. Esta crema se puede aplicar en todo el cuerpo, teniendo un efecto similar en cada zona en la que se aplica. Este bálsamo permite fortalecer la capa natural de la piel, al tiempo que ofrece proteínas y demás nutrientes. Puede ser usado de forma complementaria en personas que requieran atención médica para su piel. No está contraindicado en casos de psoriasis, diabetes y piel madura. Por su parte, el gel de baño es suave y reparador para pieles secas y muy secas. Tiene un efecto considerablemente rápido sobre la piel afectada. Se utiliza para mejorar los síntomas de la resequedad corporal. Aporta minerales y demás nutrientes al cuerpo. Es completamente de uso externo, por lo que no se recomienda colocar en otras cavidades del cuerpo. Este gel de baño enriquecido con urea, también contiene lactato, una sustancia responsable de la hidratación corporal. Estos productos están testados dermatológicamente y no representan un riesgo para la salud. Son hipoalergénicos., Modo de implementación: Te recomendamos aplicar sobre la piel, masajeando suavemente hasta hacer espuma. Aclarar con abundante agua.IndicacionesIndicado para limpiar la piel de agentes contaminantes que están presentes en el medio ambiente. ContraindicacionesNo utilizar en personas sensibles a sus componentes.No utilizar en niños menores de 2 años, en caso de ser necesaria su aplicación, consultar previamente al pediatra o farmacéutico.Usar con precaución en pieles sensibles.Conservar en un lugar fresco y seco.Mantener alejado de fuentes de luz y calor.Mantener alejado del alcance de los niños., Distancia: 0.4046472548438982
    Nombre: MELASES CYSTEAMINE CR GEL 50ML, Descripción: Sesderma Melases Cysteamine es una crema gel para piel con hiperpigmentación, tendencia a melasma o signos de fotoenvejecimiento. Apta para todo tipo de pieles y recomendada para fototipos altos. Fototipo es el término que se utiliza para describir la respuesta de la piel a la exposición solar., Modo de implementación: Con el rostro limpio y seco, aplica 3 o 4 pulsaciones en los dedos y masajea rostro y cuello hasta su completa absorción. También puede usarse en axilas, codos, piernas, manos y brazos. Evita el contacto con los ojos. Se recomienda uso diario, mañana y noche., Distancia: 0.4112510990089042
    Nombre: CREMA REAFIRMANTE 500 ML, Descripción: CREMA REAFIRMANTE ESPECIAL FLACIDEZ 500 ML. Reafirmante concentrado unisex que hidrata y regenera piel, suavizando los signos de la flacidez cutánea y dotándole de un aspecto más liso y tonificado. Apto para todo tipo de pieles. Textura fluida y ligera y de fácil absorción, resulta ideal su uso para todo el cuerpo. No deja sensación grasa. No testado en animales., Modo de implementación: Aplicar preferentemente después del baño o ducha, mediante suaves masajes en todo el cuerpo., Distancia: 0.3855254374369854""",
    """Nombre: PEPTIVIS LIMON 20 SOBRES, Descripción: Peptivis® es un complemento alimenticio a base de péptidos bioactivos de colágeno hidrolizado, HMB, Leucina y Vitamina D3. Ayuda a aumentar la masa muscular , mejorar la fuerza muscular , reducir la pérdida de capacidad motora y prevenir la pérdida de masa muscular y la sarcopenia., Modo de implementación: 2 sobres al día. Disolver el contenido del sobre en 150-200 ml de agua y mezclar bien., Distancia: 0.4125694536357126
    Nombre: K2 + D3+ SILICIO 60 COMP, Descripción: K2 + D3 + Silicio de Natysal es un complemento alimenticio con un alto contenido de Vitamina K2 y D3, con Acido ortosilícico estabilizado con colina, Modo de implementación: Se recomienda tomar 1 comprimido al día, ingiriéndolo o disolviéndolo en la boca, Distancia: 0.38284188623097903
    Nombre: Duplo Aquilea Colágeno + Magnesio, 2 x 375 g, Descripción: Duplo Aquilea Colágeno + Magnesio destaca por su contenido en:Magnesio, que contribuye al funcionamiento normal de los músculos, al mantenimiento de los huesos en condiciones normales y al metabolismo energético normal.Vitamina C, que contribuye a la formación normal de colágeno para el funcionamiento normal de los huesos y de os cartílagos.Sabor a limón., Modo de implementación: Te recomendamos tomar 1 cucharada de 12,5 g (un cacito) al día disuelto en un vaso de agua.IndicacionesIndicado para adultos.ContraindicacionesMantener fuera del alcance de los niños.Conservar en un lugar fresco y seco.No superar la dosis diaria recomendada.Los complementos alimenticios no deben sustituir una dieta variada y saludable.No consumir una vez pasada la fecha de caducidad que aparece en el envase., Distancia: 0.3762568670490928
    Nombre: BIG VITAMINA D3 + K2 -120 VEGICAPS, Descripción: Sinergia de dos nutrientes clave para la salud ósea y cardiovascular. Apto para vegetarianos., Modo de implementación: Tomar una VegCap al día con la comida o con un vaso de agua., Distancia: 0.3502689031446813
    Nombre: VIGOR SOL ACTIF PLUS PERLAS, Descripción: Complemento alimenticio con aceite de onagra y vitaminas (A,C,B2,B3,B8) y minerales como el zinc y el cobre, que contribuyen al mantenimiento de la piel en condiciones normales., Modo de implementación: Tomar 1 perla, a cualquier hora del día., Distancia: 0.39762914847604014""",
    """Nombre: SOLARAY PIEL CABELLO Y UÑAS 60 CAPSULAS, Descripción: Solaray Piel, Cabello y Uñas 60 Cápsulas Vegetales. Solaray Pelo, Piel y Uñas te ofrece la solución perfecta. Esta fórmula única contiene una mezcla de aminoácidos, minerales, vitaminas y antioxidantes que proporcionan los nutrientes esenciales a tu cabello, uñas y piel., Modo de implementación: Tomar una Vegcap al día, con la comida o con un vaso de agua., Distancia: 0.3916230192519107
    Nombre: Pack Farline con Árbol de Té Champú+Spray, 1 Ud, Descripción: Pack Farline con Árbol de Té Champú+Spray contiene los esenciales para el cuidado del cabello de los más pequeños con un agradable perfume a fresa. El spray ayuda a facilitar el peinado y el champú es de uso diario., Modo de implementación: Utilizar el champú sobre el cabello, realizando un masaje sobre el cuero cabelludo hasta que aparezca espuma. Aclarar con abundante agua. Aplicar varias pulverizaciones del spray sobre el cabello limpio y húmedo, cepillando con un peine para desenredar y extender el producto.IndicacionesIndicado para el cuidado del cabello de los niños.ContraindicacionesEvitar el contacto con los ojos.No ingerir.Mantener fuera del alcance de los niños. Conservar en un lugar fresco y seco., Distancia: 0.4421944964566139
    Nombre: CURLY METHOD PACK, Descripción: CURLY METHOD PACK. Este pack contiene:  - Champú Final Wash: se utilizará las primeras 5 veces que apliques la rutina. Este champú es el único de la línea que contiene sulfatos, estos son necesarios para eliminar todos los residuos que se han ido depositando en nuestro cabello. Después de cada lavado de Final Wash, continuaremos con el paso 2, 3 y 4 de la rutina, para ver los efectos desde el primer día.  Una vez finalizadas las 5 primeras rutinas, sustituiremos el Final Wash, por el Champú Low-Poo, que será el champú definitivo para el resto de lavados.  - Mascarilla Co-wash: Hidrata, repara y desenreda el cabello. Ayuda a controlar el encrespamiento y a reducir la sequedad dejando los rizos suaves y brillantes.  - Crema de peinado Leave-in: Sin aclarado. Repara el cabello y elimina el efecto frizz. Ayuda a definir los rizos sin apelmazar.  - Activador de rizos Styling: Gel de definición que da forma, resalta y revitaliza los rizos con un aspecto natural. Fijación suave sin apelmazar., Modo de implementación: Aplicar sobre el cabello mojado el champú Final Wash (se utilizará las primeras 5 veces que apliques la rutina. Una vez finalizadas las 5 primeras rutinas, sustituiremos el Final Wash, por el Champú Low-Poo). Enjabonar bien y masajear el todo el cuero cabelludo. Aclarar con abundante agua.  Continuar con la Mascarilla Curly para una hidratación profunda.  Para evitar el encrespamiento y que nuestros rizos tengan un aspecto revitalizado y definido, aplicar la crema reparadora sin aclarado.  Tras el uso de la crema de peinado y con el cabello aún húmedo, aplicar el gel repartiéndolo y dándole forma al rizo con las manos. Por último, secar al aire o usar difusor a baja temperatura, tocando el cabello lo menos posible, para evitar frizz o encrespamiento., Distancia: 0.3825830042098063
    Nombre: NUK Cepillo Extrasuave, 1 Unidad, Descripción: Cepillo para peinar a los más pequeños, fabricado con cerdas de pelo 100% natural. Son extrasuaves, lo que permite un masaje del cuero cabelludo delicado. Su mango es antideslizante. Es perfecto para bebés recién nacidos.El color dependerá del stock disponible en la farmacia., Modo de implementación: Masajear suavemente la cabeza del bebé.IndicacionesPara cepillar y masajear el cuero cabelludo de los bebés.ContraindicacionesMantener alejado de fuentes de calor. No utilizar si alguno de sus componentes está dañado o en malas condiciones, Distancia: 0.43408808472099814
    Nombre: CREMA REAFIRMANTE 500 ML, Descripción: CREMA REAFIRMANTE ESPECIAL FLACIDEZ 500 ML. Reafirmante concentrado unisex que hidrata y regenera piel, suavizando los signos de la flacidez cutánea y dotándole de un aspecto más liso y tonificado. Apto para todo tipo de pieles. Textura fluida y ligera y de fácil absorción, resulta ideal su uso para todo el cuerpo. No deja sensación grasa. No testado en animales., Modo de implementación: Aplicar preferentemente después del baño o ducha, mediante suaves masajes en todo el cuerpo., Distancia: 0.44209098220027265""",
    """Nombre: SCHUSSLER AC FAC AVENA 25ML, Descripción: Aceite facial intensamente nutritivo y calmante para una piel aterciopelada, suave y resplandeciente, recomendado para todo tipo de piel. Con una fórmula repleta de valiosos aceites vegetales, escualano y dos tipos de sales tisulares de Schüssler para hidratar y suavizar la piel de forma eficaz y protegerla del daño externo. Especialmente recomendado para pieles sensibles con tendencia a la rosácea. Incorpóralo a tu rutina diaria de cuidado de la piel. Con una fórmula ligera que no dejará residuos grasos en la piel., Modo de implementación: Masajee suavemente unas gotas en la piel después de celarla completamente y antes de pasar al siguiente paso de su rutina, por ejemplo, aplicar una crema facial hidratante y nutritiva., Distancia: 0.36627977226420894
    Nombre: Pack Eucerin Ph5 Bálsamo Nutritivo + Oleogel Ducha, 450 ml + 400 ml, Descripción: Pack Eucerin Ph5 Bálsamo Nutritivo + Oleogel Ducha es un pack con un contenido neto total de 850 ml de producto. En este pack viene oleogel de ducha, el cual está indicado para el cuidado de las pieles sensibles y secas. Este producto ayuda a prevenir la sequedad y descamación cutánea. También colabora con la preservación del manto ácido protector de la piel. Su fórmula ayuda a incrementar los lípidos de la superficie de la piel hasta un mínimo de 140 %. Viene en una presentación con 400 ml, los cuales son más que suficientes para observar resultados en la piel. El resultado del uso constante de este producto es una piel limpia y protegida frente a las agresiones externas.El otro producto del pack es un bálsamo para pieles sensibles. Es muy suave y efectivo para la cara y el cuerpo, ya que contribuye a la regeneración de las defensas naturales de la piel. Este bálsamo se ha desarrollado para ser aplicado sobre la piel corporal y facial sensible. La fórmula contiene ciertos ingredientes activos que estimulan la regeneración de la piel y fortalece la barrera protectora de la misma. También defiende la piel frente a la irritación, ayuda a restaurar los niveles de pH y repone las reservas de hidratantes de la propia piel hasta por 24 horas., Modo de implementación: Te recomendamos aplicar el oleogel durante la ducha y después de la misma el bálsamo.IndicacionesIndicado para lograr un cuidado e hidratación óptima de la piel del cuerpo y cara.ContraindicacionesNo utilizar en personas sensibles a sus componentes.No utilizar en niños menores de 2 años, en caso de ser necesaria su aplicación, consultar previamente al pediatra o farmacéutico.Conservar en un lugar fresco y seco.No ingerir.Evitar el contacto con los ojos.Mantener alejado de fuentes de luz y calor., Distancia: 0.3503244068166941
    Nombre: Pack Eucerin Urea Repair Plus Bálsamo Nutritivo + Gel Baño, 450 ml + 400 ml, Descripción: Pack Eucerin Urea Repair Plus Bálsamo Nutritivo + Gel Baño con un poderoso efecto si se utiliza en conjunto. El bálsamo nutritivo tiene una acción aproximada de 48 horas. Se encarga de revitalizar y suavizar la piel. Una vez aplicado este producto, no permite que regresen los síntomas de la piel seca y deshidratada. Esta crema se puede aplicar en todo el cuerpo, teniendo un efecto similar en cada zona en la que se aplica. Este bálsamo permite fortalecer la capa natural de la piel, al tiempo que ofrece proteínas y demás nutrientes. Puede ser usado de forma complementaria en personas que requieran atención médica para su piel. No está contraindicado en casos de psoriasis, diabetes y piel madura. Por su parte, el gel de baño es suave y reparador para pieles secas y muy secas. Tiene un efecto considerablemente rápido sobre la piel afectada. Se utiliza para mejorar los síntomas de la resequedad corporal. Aporta minerales y demás nutrientes al cuerpo. Es completamente de uso externo, por lo que no se recomienda colocar en otras cavidades del cuerpo. Este gel de baño enriquecido con urea, también contiene lactato, una sustancia responsable de la hidratación corporal. Estos productos están testados dermatológicamente y no representan un riesgo para la salud. Son hipoalergénicos., Modo de implementación: Te recomendamos aplicar sobre la piel, masajeando suavemente hasta hacer espuma. Aclarar con abundante agua.IndicacionesIndicado para limpiar la piel de agentes contaminantes que están presentes en el medio ambiente. ContraindicacionesNo utilizar en personas sensibles a sus componentes.No utilizar en niños menores de 2 años, en caso de ser necesaria su aplicación, consultar previamente al pediatra o farmacéutico.Usar con precaución en pieles sensibles.Conservar en un lugar fresco y seco.Mantener alejado de fuentes de luz y calor.Mantener alejado del alcance de los niños., Distancia: 0.35946430888261427
    Nombre: CREMA REAFIRMANTE 500 ML, Descripción: CREMA REAFIRMANTE ESPECIAL FLACIDEZ 500 ML. Reafirmante concentrado unisex que hidrata y regenera piel, suavizando los signos de la flacidez cutánea y dotándole de un aspecto más liso y tonificado. Apto para todo tipo de pieles. Textura fluida y ligera y de fácil absorción, resulta ideal su uso para todo el cuerpo. No deja sensación grasa. No testado en animales., Modo de implementación: Aplicar preferentemente después del baño o ducha, mediante suaves masajes en todo el cuerpo., Distancia: 0.33572734167627827
    Nombre: CREMA ACEITE ROSA MOSQU 50ML, Descripción: "CREMA DE ACEITE ROSA MOSQUETA. Presentado en un envase de 50 ml. Nutre, regenera, repara y protege la piel, dando una sensación inmediata de suavidad. Contribuye a reducir las arrugas. Mejora el estado de las cicatrices. Aporta a la piel la hidratación y emoliencia necesarias para evitar la sequedad cutánea. Protege de agresiones externas como las variaciones de humedad y temperatura ambientales. Favorece la rápida recuperación del equilibrio fisiológico de la piel.  Sin parabenos, no testado en animales.  Precauciones: Evitar contacto con los ojos y mucosas.", Modo de implementación: Aplicar dos veces al día, mañana y noche, después de limpiar y tonificar la piel. Repartir la crema por la cara, cuello y escote dando un suave masaje con las yemas de los dedos hasta su completa absorción., Distancia: 0.37240970352958747""",
    """Nombre: Duplo Farline Desodorante Spray Sensible, 2 Uds, Descripción: Farline Desodorante Spray Sensible no contiene sales de aluminio y aporta hasta 24 horas de protección. Apto para pieles sensibles.Sin alcohol.Testado dermatologicamente., Modo de implementación: Te recomendamos aplicar Duplo Farline Desodorante Spray Sensible manteniendo el envase en posición vertical y vaporizar a una distancia de 15 cm de la axila.IndicacionesIndicado para adultos.ContraindicacionesMantener fuera del alcance de los niños.Conservar en un lugar fresco y seco.Evitar el contacto con los ojos y la boca.No ingerir., Distancia: 0.47837449336449844
    Nombre: Duplo Farline Desodorante Spray Extra_Dry, 2 Uds, Descripción: Farline Desodorante Spray Extra-Dry ayuda a controlar la sudoración excesiva, absorbiendo el exceso de humedad y aportando una máxima protección.Sin alcohol.Testado bajo control dermatológico., Modo de implementación: Te recomendamos aplicar Farline Desodorante Spray Extra-Dry manteniendo el envase en posición vertical y vaporizar a una distancia de 15 cm de la axila. Dejar secar completamente antes de vestir.IndicacionesIndicado para adultos.ContraindicacionesMantener fuera del alcance de los niños.Conservar en un lugar fresco y seco.Evitar el contacto con los ojos y la boca.No ingerir., Distancia: 0.472670669531567
    Nombre: Bach Rescue Spray, 20 ml, Descripción: Este spray está formulado a base de CherryPlum, Clematis, Impatiens, Rock Rose y Starof Bethelem. Esta combinación resulta efectiva para momentos de crisis nerviosas, estados de shock y angustia profunda. Ayuda también a encontrar la calma ante situaciones que van a suponer un estado de ansiedad o estrés., Modo de implementación: Pulverizar 2 veces sobre la lengua.IndicacionesIndicado para personas que necesitan alivio ante momentos de ansiedad o estrés.ContraindicacionesMantener fuera del alcance de los niños. Conservar en un lugar fresco y seco. Evitar el contacto con los ojos., Distancia: 0.45975424701834544
    Nombre: Duplo Farline Desodorante para Hombre en Spray, 2 x 150 ml, Descripción: duplo farline desodorante para hombre en spray, ayuda al control de la sudoración excesiva, absorbiendo el exceso de humedad y ofreciendo una alta protección durante 48 horas. contiene alantoína y vitamina e.cuenta con una fórmula anti_manchas para la ropa. no contiene alcohol. cuida de la piel del hombre, incluso de la más sensible. está testado bajo control dermatológico., Modo de implementación: te recomendamos utilizar el desodorante tras la ducha, sobre la piel limpia y seca, pulverizando durante dos segundos a unos 15 cm de la axila.Indicacionesindicado para la higiene personal del hombre. Contraindicacionesmantener alejado de fuentes de calor.conservar en un lugar fresco y seco. no ingerir.evitar el contacto con los ojos y mucosas.mantener alejado del alcance de los niños., Distancia: 0.47551133082408104
    Nombre: Pack Bexident Fresh Breath Colutorio + Spray, Descripción: La gama Fresh Breath de Bexident ayuda a combatir el mal aliento neutralizándolo y combatiendo los sulfuros volátiles que lo causan. Además, alivia la sequedad bucal y tiene propiedades antisépticas., Modo de implementación: Se recomienda usar el colutorio 3 veces al día, después del cepillado y durante, al menos, 30 segundos. Se recomienda pulverizar el spray 3 o 4 veces en la boca y no enjuagar.IndicacionesIndicado para conseguir una buena higiene bucal y un aliento fresco y sano.ContraindicacionesEvitar el contacto con los ojos. No ingerir. Mantener fuera del alcance de los niños., Distancia: 0.44051285926191364""",
]

# Respuestas generadas por el sistema RAG
generated_answers_by_gemini = [
    """{'type': 'product_search', 'message': '¡Hola! He encontrado algunas cremas que podrían ser útiles para las estrías:\n\n* **CREMA ACEITE ROSA MOSQU 50ML:** Esta crema contiene aceite de rosa mosqueta, conocido por sus propiedades regenerativas y cicatrizantes. Puede ayudar a mejorar la apariencia de las estrías, haciéndolas menos visibles. Se recomienda aplicar la crema dos veces al día, masajeando suavemente sobre la zona afectada.\n* **Pack Uresim Serum Ác.Hialurónico 30 ml + Crema Nutri+ 50 ml:** Este pack contiene un serum con ácido hialurónico, que hidrata la piel en profundidad, y una crema nutritiva que ayuda a mejorar la elasticidad de la piel. Ambos productos pueden ser beneficiosos para las estrías, ya que ayudan a mantener la piel hidratada y flexible. Se recomienda aplicar el serum por la mañana y la crema por la noche.\n* **Pack Eucerin Urea Repair Plus Bálsamo Nutritivo + Gel Baño, 450 ml + 400 ml:** Este pack contiene un bálsamo nutritivo y un gel de baño, ambos con urea, que ayuda a hidratar y suavizar la piel. La urea también puede ayudar a mejorar la apariencia de las estrías. Se recomienda aplicar el bálsamo después del baño o la ducha, masajeando suavemente sobre la zona afectada.\n* **MELASES CYSTEAMINE CR GEL 50ML:** Este gel contiene cisteamina, un ingrediente que ayuda a estimular la producción de colágeno y elastina, lo que puede ayudar a mejorar la apariencia de las estrías. Se recomienda aplicar el gel dos veces al día, masajeando suavemente sobre la zona afectada.\n* **CREMA REAFIRMANTE 500 ML:** Esta crema está diseñada para reafirmar la piel y mejorar su elasticidad. Puede ser útil para las estrías, ya que ayuda a mejorar la apariencia de la piel y a reducir la visibilidad de las estrías. Se recomienda aplicar la crema dos veces al día, masajeando suavemente sobre la zona afectada.\n\nRecuerda que los resultados pueden variar de persona a persona y es importante consultar con un médico o farmacéutico para obtener una recomendación personalizada. \n', 'products': [{'codigo_web': '035120', 'nombre': 'CREMA ACEITE ROSA MOSQU 50ML', 'codigo_nacional': '0351204', 'descripcion': '"CREMA DE ACEITE ROSA MOSQUETA. Presentado en un envase de 50 ml. Nutre, regenera, repara y protege la piel, dando una sensación inmediata de suavidad. Contribuye a reducir las arrugas. Mejora el estado de las cicatrices. Aporta a la piel la hidratación y emoliencia necesarias para evitar la sequedad cutánea. Protege de agresiones externas como las variaciones de humedad y temperatura ambientales. Favorece la rápida recuperación del equilibrio fisiológico de la piel. \xa0Sin parabenos, no testado en animales. \xa0Precauciones: Evitar contacto con los ojos y mucosas."', 'modo_implementacion': 'Aplicar dos veces al día, mañana y noche, después de limpiar y tonificar la piel. Repartir la crema por la cara, cuello y escote dando un suave masaje con las yemas de los dedos hasta su completa absorción.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/035120.jpg', 'distance_to_query': 0.3748334345316666}, {'codigo_web': '017693', 'nombre': 'Pack Uresim Serum Ác.Hialurónico 30 ml + Crema Nutri+ 50 ml', 'codigo_nacional': '0176937', 'descripcion': 'Pack Uresim Serum Ác.Hialurónico y\xa0Crema Nutri+El serum antiedad de Uresim con una alta concentración de ácido hialurónico, constituye un tratamiento integral para ayudar a la hidratación de la piel y prevenir el envejecimiento prematuro. Contribuye a prevenir las arrugas y líneas de expresión.Además ayuda a aportar tersura y suavidad a la piel. La Crema Nutri+ es nutritiva y está indicada para pieles secas y maduras. Contiene alta concentración de aceites vegetales, como el aceite de Rosa Mosqueta, aceite de Macadamia, manteca de Karité y aceite de Soja. Esta formulación ayuda a aportar nutrientes esenciales como ácidos grasos, isoflavonas de soja y\xa0 vitaminas\xa0 A, C y E a la piel.\xa0El ácido hialurónico y el extracto de caviar contribuye a la microcirculación t la oxigenación de la piel.', 'modo_implementacion': 'Te recomendamos que la Uresim Pure Hyaluronic Acid Serum lo apliques por la mañana y/o noche sobre piel limpia y seca con un suave masaje hasta su total absorción. Puede empelarse como base de día o como reparador nocturno.La Crema Nutri+ Uresim la puedes aplicar día y/o noche.IndicacionesIndicado para adultos.ContraindicacionesEste tratamiento está especialmente indicado para pieles secas, deshidratadas y con aspecto cansado y apagado.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/017693.jpg', 'distance_to_query': 0.40913467599995934}, {'codigo_web': '013589', 'nombre': 'Pack Eucerin Urea Repair Plus Bálsamo Nutritivo + Gel Baño, 450 ml + 400 ml', 'codigo_nacional': '0135897', 'descripcion': 'Pack Eucerin Urea Repair Plus Bálsamo Nutritivo + Gel Baño con un poderoso efecto si se utiliza en conjunto. El bálsamo nutritivo tiene una acción aproximada de 48 horas. Se encarga de revitalizar y suavizar la piel. Una vez aplicado este producto, no permite que regresen los síntomas de la piel seca y deshidratada. Esta crema se puede aplicar en todo el cuerpo, teniendo un efecto similar en cada zona en la que se aplica. Este bálsamo permite fortalecer la capa natural de la piel, al tiempo que ofrece proteínas y demás nutrientes. Puede ser usado de forma complementaria en personas que requieran atención médica para su piel. No está contraindicado en casos de psoriasis, diabetes y piel madura. Por su parte, el gel de baño es suave y reparador para pieles secas y muy secas. Tiene un efecto considerablemente rápido sobre la piel afectada. Se utiliza para mejorar los síntomas de la resequedad corporal. Aporta minerales y demás nutrientes al cuerpo. Es completamente de uso externo, por lo que no se recomienda colocar en otras cavidades del cuerpo. Este gel de baño enriquecido con urea, también contiene lactato, una sustancia responsable de la hidratación corporal. Estos productos están testados dermatológicamente y no representan un riesgo para la salud. Son hipoalergénicos.', 'modo_implementacion': 'Te recomendamos aplicar sobre la piel, masajeando suavemente hasta hacer espuma. Aclarar con abundante agua.IndicacionesIndicado para limpiar la piel de agentes contaminantes que están presentes en el medio ambiente. ContraindicacionesNo utilizar en personas sensibles a sus componentes.No utilizar en niños menores de 2 años, en caso de ser necesaria su aplicación, consultar previamente al pediatra o farmacéutico.Usar con precaución en pieles sensibles.Conservar en un lugar fresco y seco.Mantener alejado de fuentes de luz y calor.Mantener alejado del alcance de los niños.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/013589.jpg', 'distance_to_query': 0.4046472548438982}, {'codigo_web': '026648', 'nombre': 'MELASES CYSTEAMINE CR GEL 50ML', 'codigo_nacional': '0266485', 'descripcion': 'Sesderma Melases Cysteamine es una crema gel para piel con hiperpigmentación, tendencia a melasma o signos de fotoenvejecimiento. Apta para todo tipo de pieles y recomendada para fototipos altos. Fototipo es el término que se utiliza para describir la respuesta de la piel a la exposición solar.', 'modo_implementacion': 'Con el rostro limpio y seco, aplica 3 o 4 pulsaciones en los dedos y masajea rostro y cuello hasta su completa absorción. También puede usarse en axilas, codos, piernas, manos y brazos. Evita el contacto con los ojos. Se recomienda uso diario, mañana y noche.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/026648.jpg', 'distance_to_query': 0.4112510990089042}, {'codigo_web': '033814', 'nombre': 'CREMA REAFIRMANTE 500 ML', 'codigo_nacional': '0338144', 'descripcion': 'CREMA REAFIRMANTE ESPECIAL FLACIDEZ 500 ML. Reafirmante concentrado unisex que hidrata y regenera piel, suavizando los signos de la flacidez cutánea y dotándole de un aspecto más liso y tonificado. Apto para todo tipo de pieles. Textura fluida y ligera y de fácil absorción, resulta ideal su uso para todo el cuerpo. No deja sensación grasa. No testado en animales.', 'modo_implementacion': 'Aplicar preferentemente después del baño o ducha, mediante suaves masajes en todo el cuerpo.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/033814.jpg', 'distance_to_query': 0.3855254374369854}]}""",
    """{'type': 'product_search', 'message': '¡Hola! He encontrado algunos suplementos de vitamina D que podrían ser adecuados para personas mayores:\n\n* **BIG VITAMINA D3 + K2 -120 VEGICAPS:** Este suplemento combina vitamina D3 y vitamina K2, ambas importantes para la salud ósea. La vitamina D3 ayuda a la absorción del calcio, mientras que la vitamina K2 contribuye a la formación de hueso. Se recomienda tomar una cápsula al día, con una comida.\n* **VIGOR SOL ACTIF PLUS PERLAS:** Este suplemento contiene vitamina D3, vitamina K2 y otros nutrientes como el magnesio y el zinc, que también son importantes para la salud ósea. Se recomienda tomar una perla al día, con una comida.\n* **K2 + D3+ SILICIO 60 COMP:** Este suplemento combina vitamina D3, vitamina K2 y silicio, un mineral que contribuye a la salud de los huesos, el cabello y la piel. Se recomienda tomar una comprimido al día, con una comida.\n\nEs importante recordar que la dosis adecuada de vitamina D puede variar según la edad, el estilo de vida y la salud general de la persona. Es recomendable consultar con un médico o farmacéutico para determinar la dosis adecuada para cada caso.\n\nRecuerda que estos suplementos no son un sustituto de una dieta saludable y un estilo de vida activo. \n\nEspero que esta información te sea útil. \n', 'products': [{'codigo_web': '014337', 'nombre': 'PEPTIVIS LIMON 20 SOBRES', 'codigo_nacional': '0143373', 'descripcion': 'Peptivis® es un complemento alimenticio a base de péptidos bioactivos de colágeno hidrolizado, HMB, Leucina y Vitamina D3. Ayuda a aumentar la masa muscular , mejorar la fuerza muscular , reducir la pérdida de capacidad motora y prevenir la pérdida de masa muscular y la sarcopenia.', 'modo_implementacion': '2 sobres al día. Disolver el contenido del sobre en 150-200 ml de agua y mezclar bien.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/014337.jpg', 'distance_to_query': 0.4125694536357126}, {'codigo_web': '031061', 'nombre': 'K2 + D3+ SILICIO 60 COMP', 'codigo_nacional': '0310614', 'descripcion': 'K2 + D3 + Silicio de Natysal es un complemento alimenticio con un alto contenido de Vitamina K2 y D3, con Acido ortosilícico estabilizado con colina', 'modo_implementacion': 'Se recomienda tomar 1 comprimido al día, ingiriéndolo o disolviéndolo en la boca', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/031061.jpg', 'distance_to_query': 0.38284188623097903}, {'codigo_web': '013782', 'nombre': 'Duplo Aquilea Colágeno + Magnesio, 2 x 375 g', 'codigo_nacional': '0137822', 'descripcion': 'Duplo Aquilea Colágeno + Magnesio\xa0destaca por su contenido en:Magnesio,\xa0que contribuye al funcionamiento normal de los músculos, al mantenimiento de los huesos en condiciones normales y al metabolismo energético normal.Vitamina C,\xa0que contribuye a la formación normal de colágeno para el funcionamiento normal de los huesos y de os cartílagos.Sabor a limón.', 'modo_implementacion': 'Te recomendamos tomar 1 cucharada de 12,5 g (un cacito) al día disuelto en un vaso de agua.IndicacionesIndicado para adultos.ContraindicacionesMantener fuera del alcance de los niños.Conservar en un lugar fresco y seco.No superar la dosis diaria recomendada.Los complementos alimenticios no deben sustituir una dieta variada y saludable.No consumir una vez pasada la fecha de caducidad que aparece en el envase.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/013782.jpg', 'distance_to_query': 0.3762568670490928}, {'codigo_web': '035115', 'nombre': 'BIG VITAMINA D3 + K2 -120 VEGICAPS', 'codigo_nacional': '0351150', 'descripcion': 'Sinergia de dos nutrientes clave para la salud ósea y cardiovascular. Apto para vegetarianos.', 'modo_implementacion': 'Tomar una VegCap al día con la comida o con un vaso de agua.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/035115.jpg', 'distance_to_query': 0.3502689031446813}, {'codigo_web': '033263', 'nombre': 'VIGOR SOL ACTIF PLUS PERLAS', 'codigo_nacional': '0332630', 'descripcion': 'Complemento alimenticio con aceite de onagra y vitaminas (A,C,B2,B3,B8) y minerales como el zinc y el cobre, que contribuyen al mantenimiento de la piel en condiciones normales.', 'modo_implementacion': 'Tomar 1 perla, a cualquier hora del día.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/033263.jpg', 'distance_to_query': 0.39762914847604014}]}""",
    """{'type': 'product_search', 'message': '¡Hola! He encontrado algunos productos que podrían ser útiles para la caída del cabello:\n\n* **SOLARAY PIEL CABELLO Y UÑAS 60 CAPSULAS:** Este suplemento contiene vitaminas y minerales que contribuyen a la salud del cabello, la piel y las uñas. Puede ayudar a fortalecer el cabello y a prevenir la caída. Se recomienda tomar una cápsula al día, con una comida.\n* **Pack Farline con Árbol de Té Champú+Spray, 1 Ud:** Este pack contiene un champú y un spray con aceite de árbol de té, que tiene propiedades antibacterianas y antifúngicas. Puede ayudar a controlar la caspa y a prevenir la caída del cabello causada por infecciones del cuero cabelludo. Se recomienda usar el champú dos o tres veces por semana y el spray después de cada lavado.\n\nLos otros productos que mencionaste no son directamente relevantes para la caída del cabello:\n\n* **CURLY METHOD PACK:** Este pack está diseñado para el cuidado del cabello rizado y no tiene propiedades específicas para la caída del cabello.\n* **NUK Cepillo Extrasuave, 1 Unidad:** Este cepillo es para bebés y no tiene propiedades para la caída del cabello.\n* **CREMA REAFIRMANTE 500 ML:** Esta crema está diseñada para reafirmar la piel y no tiene propiedades para la caída del cabello.\n\nRecuerda que la caída del cabello puede tener diferentes causas, y es importante consultar con un médico o dermatólogo para determinar la causa específica y obtener un tratamiento adecuado. \n\nEspero que esta información te sea útil. \n', 'products': [{'codigo_web': '015577', 'nombre': 'SOLARAY PIEL CABELLO Y UÑAS 60 CAPSULAS', 'codigo_nacional': '0155772', 'descripcion': 'Solaray Piel, Cabello y Uñas 60 Cápsulas Vegetales. Solaray Pelo, Piel y Uñas te ofrece la solución perfecta. Esta fórmula única contiene una mezcla de aminoácidos, minerales, vitaminas y antioxidantes que proporcionan los nutrientes esenciales a tu cabello, uñas y piel.', 'modo_implementacion': 'Tomar una Vegcap al día, con la comida o con un vaso de agua.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/015577.jpg', 'distance_to_query': 0.3916230192519107}, {'codigo_web': '012998', 'nombre': 'Pack Farline con Árbol de Té Champú+Spray, 1 Ud', 'codigo_nacional': '0129988', 'descripcion': 'Pack Farline con Árbol de Té Champú+Spray\xa0contiene los esenciales para el cuidado del cabello de los más pequeños con un agradable perfume a fresa. El spray ayuda a facilitar el peinado y el champú es de uso diario.', 'modo_implementacion': 'Utilizar el champú sobre el cabello, realizando un masaje sobre el cuero cabelludo hasta que aparezca espuma. Aclarar con abundante agua. Aplicar varias pulverizaciones del spray sobre el cabello limpio y húmedo, cepillando con un peine para desenredar y extender el producto.IndicacionesIndicado para el cuidado del cabello de los niños.ContraindicacionesEvitar el contacto con los ojos.No ingerir.Mantener fuera del alcance de los niños. Conservar en un lugar fresco y seco.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/012998.jpg', 'distance_to_query': 0.4421944964566139}, {'codigo_web': '033379', 'nombre': 'CURLY METHOD PACK', 'codigo_nacional': '0333798', 'descripcion': 'CURLY METHOD PACK. Este pack contiene: \xa0- Champú Final Wash: se utilizará las primeras 5 veces que apliques la rutina. Este champú es el único de la línea que contiene sulfatos, estos son necesarios para eliminar todos los residuos que se han ido depositando en nuestro cabello. Después de cada lavado de Final Wash, continuaremos con el paso 2, 3 y 4 de la rutina, para ver los efectos desde el primer día. \xa0Una vez finalizadas las 5 primeras rutinas, sustituiremos el Final Wash, por el Champú Low-Poo, que será el champú definitivo para el resto de lavados. \xa0- Mascarilla Co-wash: Hidrata, repara y desenreda el cabello. Ayuda a controlar el encrespamiento y a reducir la sequedad dejando los rizos suaves y brillantes. \xa0- Crema de peinado Leave-in: Sin aclarado. Repara el cabello y elimina el efecto frizz. Ayuda a definir los rizos sin apelmazar. \xa0- Activador de rizos Styling: Gel de definición que da forma, resalta y revitaliza los rizos con un aspecto natural. Fijación suave sin apelmazar.', 'modo_implementacion': 'Aplicar sobre el cabello mojado el champú Final Wash (se utilizará las primeras 5 veces que apliques la rutina. Una vez finalizadas las 5 primeras rutinas, sustituiremos el Final Wash, por el Champú Low-Poo). Enjabonar bien y masajear el todo el cuero cabelludo. Aclarar con abundante agua. \xa0Continuar con la Mascarilla Curly para una hidratación profunda. \xa0Para evitar el encrespamiento y que nuestros rizos tengan un aspecto revitalizado y definido, aplicar la crema reparadora sin aclarado. \xa0Tras el uso de la crema de peinado y con el cabello aún húmedo, aplicar el gel repartiéndolo y dándole forma al rizo con las manos. Por último, secar al aire o usar difusor a baja temperatura, tocando el cabello lo menos posible, para evitar frizz o encrespamiento.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/033379.jpg', 'distance_to_query': 0.3825830042098063}, {'codigo_web': '013644', 'nombre': 'NUK Cepillo Extrasuave, 1 Unidad', 'codigo_nacional': '0136443', 'descripcion': 'Cepillo para peinar a los más pequeños, fabricado con cerdas de pelo 100% natural. Son extrasuaves, lo que permite un masaje del cuero cabelludo delicado.\xa0Su mango es antideslizante. Es perfecto para bebés recién nacidos.El color dependerá del stock disponible en la farmacia.', 'modo_implementacion': 'Masajear suavemente la cabeza del bebé.IndicacionesPara cepillar y masajear el cuero cabelludo de los bebés.ContraindicacionesMantener alejado de fuentes de calor. No utilizar si alguno de sus componentes está dañado o en malas condiciones', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/013644.jpg', 'distance_to_query': 0.43408808472099814}, {'codigo_web': '033814', 'nombre': 'CREMA REAFIRMANTE 500 ML', 'codigo_nacional': '0338144', 'descripcion': 'CREMA REAFIRMANTE ESPECIAL FLACIDEZ 500 ML. Reafirmante concentrado unisex que hidrata y regenera piel, suavizando los signos de la flacidez cutánea y dotándole de un aspecto más liso y tonificado. Apto para todo tipo de pieles. Textura fluida y ligera y de fácil absorción, resulta ideal su uso para todo el cuerpo. No deja sensación grasa. No testado en animales.', 'modo_implementacion': 'Aplicar preferentemente después del baño o ducha, mediante suaves masajes en todo el cuerpo.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/033814.jpg', 'distance_to_query': 0.44209098220027265}]}""",
    """{'type': 'product_search', 'message': '¡Hola! He encontrado algunas cremas hidratantes que podrían ser adecuadas para piel sensible:\n\n* **SCHUSSLER AC FAC AVENA 25ML:** Esta crema contiene avena, un ingrediente conocido por sus propiedades calmantes y antiinflamatorias. Es ideal para pieles sensibles, secas o irritadas. Se recomienda aplicar la crema dos veces al día, masajeando suavemente sobre la piel limpia.\n* **Pack Eucerin Ph5 Bálsamo Nutritivo + Oleogel Ducha, 450 ml + 400 ml:** Este pack contiene un bálsamo nutritivo y un gel de ducha, ambos formulados con pH5, que ayuda a mantener el equilibrio natural de la piel. Son ideales para pieles sensibles y secas. Se recomienda aplicar el bálsamo después del baño o la ducha, masajeando suavemente sobre la piel limpia.\n* **Pack Eucerin Urea Repair Plus Bálsamo Nutritivo + Gel Baño, 450 ml + 400 ml:** Este pack contiene un bálsamo nutritivo y un gel de baño, ambos con urea, que ayuda a hidratar y suavizar la piel. La urea también puede ayudar a aliviar la picazón y la irritación. Se recomienda aplicar el bálsamo después del baño o la ducha, masajeando suavemente sobre la piel limpia.\n\nLos otros productos que mencionaste no son directamente relevantes para la piel sensible:\n\n* **CREMA REAFIRMANTE 500 ML:** Esta crema está diseñada para reafirmar la piel y no tiene propiedades específicas para la piel sensible.\n* **CREMA ACEITE ROSA MOSQU 50ML:** Esta crema contiene aceite de rosa mosqueta, que puede ser irritante para algunas pieles sensibles.\n\nRecuerda que la piel sensible puede reaccionar de forma diferente a los productos, por lo que es importante probar una pequeña cantidad de crema en una zona discreta de la piel antes de aplicarla en todo el rostro o cuerpo. \n\nEspero que esta información te sea útil. \n\n\n', 'products': [{'codigo_web': '014319', 'nombre': 'SCHUSSLER AC FAC AVENA 25ML', 'codigo_nacional': '0143199', 'descripcion': 'Aceite facial intensamente nutritivo y calmante para una piel aterciopelada, suave y resplandeciente, recomendado para todo tipo de piel. Con una fórmula repleta de valiosos aceites vegetales, escualano y dos tipos de sales tisulares de Schüssler para hidratar y suavizar la piel de forma eficaz y protegerla del daño externo. Especialmente recomendado para pieles sensibles con tendencia a la rosácea. Incorpóralo a tu rutina diaria de cuidado de la piel. Con una fórmula ligera que no dejará residuos grasos en la piel.', 'modo_implementacion': 'Masajee suavemente unas gotas en la piel después de celarla completamente y antes de pasar al siguiente paso de su rutina, por ejemplo, aplicar una crema facial hidratante y nutritiva.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/014319.jpg', 'distance_to_query': 0.36627977226420894}, {'codigo_web': '013588', 'nombre': 'Pack Eucerin Ph5 Bálsamo Nutritivo + Oleogel Ducha, 450 ml + 400 ml', 'codigo_nacional': '0135880', 'descripcion': 'Pack Eucerin Ph5 Bálsamo Nutritivo + Oleogel Ducha es un pack con un contenido neto total de 850 ml de producto. En este pack viene oleogel de ducha, el cual está indicado para el cuidado de las pieles sensibles y secas. Este producto ayuda a prevenir la sequedad y descamación cutánea. También colabora con la preservación del manto ácido protector de la piel. Su fórmula ayuda a incrementar los lípidos de la superficie de la piel hasta un mínimo de 140 %. Viene en una presentación con 400 ml, los cuales son más que suficientes para observar resultados en la piel. El resultado del uso constante de este producto es una piel limpia y protegida frente a las agresiones externas.El otro producto del pack es un bálsamo para pieles sensibles. Es muy suave y efectivo para la cara y el cuerpo, ya que contribuye a la regeneración de las defensas naturales de la piel. Este bálsamo se ha desarrollado para ser aplicado sobre la piel corporal y facial sensible. La fórmula contiene ciertos ingredientes activos que estimulan la regeneración de la piel y fortalece la barrera protectora de la misma. También defiende la piel frente a la irritación, ayuda a restaurar los niveles de pH y repone las reservas de hidratantes de la propia piel hasta por 24 horas.', 'modo_implementacion': 'Te recomendamos aplicar el oleogel durante la ducha y después de la misma el bálsamo.IndicacionesIndicado para lograr un cuidado e hidratación óptima de la piel del cuerpo y cara.ContraindicacionesNo utilizar en personas sensibles a sus componentes.No utilizar en niños menores de 2 años, en caso de ser necesaria su aplicación, consultar previamente al pediatra o farmacéutico.Conservar en un lugar fresco y seco.No ingerir.Evitar el contacto con los ojos.Mantener alejado de fuentes de luz y calor.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/013588.jpg', 'distance_to_query': 0.3503244068166941}, {'codigo_web': '013589', 'nombre': 'Pack Eucerin Urea Repair Plus Bálsamo Nutritivo + Gel Baño, 450 ml + 400 ml', 'codigo_nacional': '0135897', 'descripcion': 'Pack Eucerin Urea Repair Plus Bálsamo Nutritivo + Gel Baño con un poderoso efecto si se utiliza en conjunto. El bálsamo nutritivo tiene una acción aproximada de 48 horas. Se encarga de revitalizar y suavizar la piel. Una vez aplicado este producto, no permite que regresen los síntomas de la piel seca y deshidratada. Esta crema se puede aplicar en todo el cuerpo, teniendo un efecto similar en cada zona en la que se aplica. Este bálsamo permite fortalecer la capa natural de la piel, al tiempo que ofrece proteínas y demás nutrientes. Puede ser usado de forma complementaria en personas que requieran atención médica para su piel. No está contraindicado en casos de psoriasis, diabetes y piel madura. Por su parte, el gel de baño es suave y reparador para pieles secas y muy secas. Tiene un efecto considerablemente rápido sobre la piel afectada. Se utiliza para mejorar los síntomas de la resequedad corporal. Aporta minerales y demás nutrientes al cuerpo. Es completamente de uso externo, por lo que no se recomienda colocar en otras cavidades del cuerpo. Este gel de baño enriquecido con urea, también contiene lactato, una sustancia responsable de la hidratación corporal. Estos productos están testados dermatológicamente y no representan un riesgo para la salud. Son hipoalergénicos.', 'modo_implementacion': 'Te recomendamos aplicar sobre la piel, masajeando suavemente hasta hacer espuma. Aclarar con abundante agua.IndicacionesIndicado para limpiar la piel de agentes contaminantes que están presentes en el medio ambiente. ContraindicacionesNo utilizar en personas sensibles a sus componentes.No utilizar en niños menores de 2 años, en caso de ser necesaria su aplicación, consultar previamente al pediatra o farmacéutico.Usar con precaución en pieles sensibles.Conservar en un lugar fresco y seco.Mantener alejado de fuentes de luz y calor.Mantener alejado del alcance de los niños.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/013589.jpg', 'distance_to_query': 0.35946430888261427}, {'codigo_web': '033814', 'nombre': 'CREMA REAFIRMANTE 500 ML', 'codigo_nacional': '0338144', 'descripcion': 'CREMA REAFIRMANTE ESPECIAL FLACIDEZ 500 ML. Reafirmante concentrado unisex que hidrata y regenera piel, suavizando los signos de la flacidez cutánea y dotándole de un aspecto más liso y tonificado. Apto para todo tipo de pieles. Textura fluida y ligera y de fácil absorción, resulta ideal su uso para todo el cuerpo. No deja sensación grasa. No testado en animales.', 'modo_implementacion': 'Aplicar preferentemente después del baño o ducha, mediante suaves masajes en todo el cuerpo.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/033814.jpg', 'distance_to_query': 0.33572734167627827}, {'codigo_web': '035120', 'nombre': 'CREMA ACEITE ROSA MOSQU 50ML', 'codigo_nacional': '0351204', 'descripcion': '"CREMA DE ACEITE ROSA MOSQUETA. Presentado en un envase de 50 ml. Nutre, regenera, repara y protege la piel, dando una sensación inmediata de suavidad. Contribuye a reducir las arrugas. Mejora el estado de las cicatrices. Aporta a la piel la hidratación y emoliencia necesarias para evitar la sequedad cutánea. Protege de agresiones externas como las variaciones de humedad y temperatura ambientales. Favorece la rápida recuperación del equilibrio fisiológico de la piel. \xa0Sin parabenos, no testado en animales. \xa0Precauciones: Evitar contacto con los ojos y mucosas."', 'modo_implementacion': 'Aplicar dos veces al día, mañana y noche, después de limpiar y tonificar la piel. Repartir la crema por la cara, cuello y escote dando un suave masaje con las yemas de los dedos hasta su completa absorción.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/035120.jpg', 'distance_to_query': 0.37240970352958747}]}""",
    """{'type': 'product_search', 'message': '¡Hola! He encontrado algunos productos que podrían ser útiles para las alergias, pero ninguno es un spray nasal específico:\n\n* **Bach Rescue Spray, 20 ml:** Este spray contiene una mezcla de flores de Bach, que se utilizan para aliviar el estrés y la ansiedad. Puede ser útil para aliviar los síntomas de alergia, como la ansiedad y la irritabilidad. Se recomienda aplicar unas gotas debajo de la lengua o en un vaso de agua, según las indicaciones del fabricante.\n\nLos otros productos que mencionaste no son relevantes para las alergias:\n\n* **Duplo Farline Desodorante Spray Sensible, 2 Uds:** Este producto es un desodorante para axilas.\n* **Duplo Farline Desodorante Spray Extra_Dry, 2 Uds:** Este producto es un desodorante para axilas.\n* **Duplo Farline Desodorante para Hombre en Spray, 2 x 150 ml:** Este producto es un desodorante para axilas.\n* **Pack Bexident Fresh Breath Colutorio + Spray:** Este pack contiene un colutorio y un spray para el aliento fresco.\n\nPara aliviar los síntomas de alergia, es importante consultar con un médico o farmacéutico para obtener una recomendación personalizada. \n\nEspero que esta información te sea útil. \n\n\n', 'products': [{'codigo_web': '035132', 'nombre': 'Duplo Farline Desodorante Spray Sensible, 2 Uds', 'codigo_nacional': '0351327', 'descripcion': 'Farline Desodorante Spray Sensible\xa0no contiene sales de aluminio y aporta hasta 24 horas de protección. Apto para pieles sensibles.Sin alcohol.Testado dermatologicamente.', 'modo_implementacion': 'Te recomendamos aplicar Duplo\xa0Farline Desodorante Spray Sensible\xa0manteniendo el envase en posición vertical y vaporizar a una distancia de 15 cm de la axila.IndicacionesIndicado para adultos.ContraindicacionesMantener fuera del alcance de los niños.Conservar en un lugar fresco y seco.Evitar el contacto con los ojos y la boca.No ingerir.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/035132.jpg', 'distance_to_query': 0.47837449336449844}, {'codigo_web': '035131', 'nombre': 'Duplo Farline Desodorante Spray Extra_Dry, 2 Uds', 'codigo_nacional': '0351310', 'descripcion': 'Farline Desodorante Spray Extra-Dry\xa0ayuda a controlar la sudoración excesiva, absorbiendo el exceso de humedad y aportando una máxima protección.Sin alcohol.Testado bajo control dermatológico.', 'modo_implementacion': 'Te recomendamos aplicar\xa0Farline Desodorante Spray Extra-Dry\xa0manteniendo el envase en posición vertical y vaporizar a una distancia de 15 cm de la axila.\xa0Dejar secar completamente antes de vestir.IndicacionesIndicado para adultos.ContraindicacionesMantener fuera del alcance de los niños.Conservar en un lugar fresco y seco.Evitar el contacto con los ojos y la boca.No ingerir.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/035131.jpg', 'distance_to_query': 0.472670669531567}, {'codigo_web': '013592', 'nombre': 'Bach Rescue Spray, 20 ml', 'codigo_nacional': '0135927', 'descripcion': 'Este spray está formulado a base de CherryPlum, Clematis, Impatiens, Rock Rose y Starof Bethelem. Esta combinación resulta efectiva para momentos de crisis nerviosas, estados de shock y angustia profunda.\xa0Ayuda también a encontrar la calma ante situaciones que van a suponer un estado de ansiedad o estrés.', 'modo_implementacion': 'Pulverizar 2 veces sobre la lengua.IndicacionesIndicado para personas que necesitan alivio ante momentos de ansiedad o estrés.ContraindicacionesMantener fuera del alcance de los niños. Conservar en un lugar fresco y seco. Evitar el contacto con los ojos.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/013592.jpg', 'distance_to_query': 0.45975424701834544}, {'codigo_web': '033515', 'nombre': 'Duplo Farline Desodorante para Hombre en Spray, 2 x 150 ml', 'codigo_nacional': '0335150', 'descripcion': 'duplo farline desodorante para hombre en spray, ayuda al control de la sudoración excesiva, absorbiendo el exceso de humedad y ofreciendo una alta protección durante 48 horas. contiene alantoína y vitamina e.cuenta con una fórmula anti_manchas para la ropa. no contiene alcohol. cuida de la piel del hombre, incluso de la más sensible. está testado bajo control dermatológico.', 'modo_implementacion': 'te recomendamos utilizar el desodorante tras la ducha, sobre la piel limpia y seca, pulverizando durante dos segundos a unos 15 cm de la axila.Indicacionesindicado para la higiene personal del hombre.\xa0Contraindicacionesmantener alejado de fuentes de calor.conservar en un lugar fresco y seco.\xa0no ingerir.evitar el contacto con los ojos y mucosas.mantener alejado del alcance de los niños.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/033515.jpg', 'distance_to_query': 0.47551133082408104}, {'codigo_web': '013042', 'nombre': 'Pack Bexident Fresh Breath Colutorio + Spray', 'codigo_nacional': '0130427', 'descripcion': 'La gama Fresh Breath de Bexident ayuda a combatir el mal aliento neutralizándolo y combatiendo los sulfuros volátiles que lo causan. Además, alivia la sequedad bucal y tiene propiedades antisépticas.', 'modo_implementacion': 'Se recomienda usar el colutorio 3 veces al día, después del cepillado y durante, al menos, 30 segundos. Se recomienda pulverizar el spray 3 o 4 veces en la boca y no enjuagar.IndicacionesIndicado para conseguir una buena higiene bucal y un aliento fresco y sano.ContraindicacionesEvitar el contacto con los ojos. No ingerir. Mantener fuera del alcance de los niños.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/013042.jpg', 'distance_to_query': 0.44051285926191364}]}"""
]

# Creación del DataFrame para el dataset de evaluación

eval_bigquery = pd.DataFrame(
    {
        "prompt": [
            "Answer the question: " + question + " Context: " + item
            for question, item in zip(questions, retrieved_contexts_by_bigquery)
        ],
        "response": retrieved_contexts_by_bigquery,
    }
)

eval_rerank = pd.DataFrame(
    {
        "prompt": [
            "Answer the question: " + question + " Context: " + item
            for question, item in zip(questions, retrieved_contexts_by_bigquery)
        ],
        "response": generated_answers_by_rerank,
    }
)

eval_gemini = pd.DataFrame(
    {
        "prompt": [
            "Answer the question: " + question + " Context: " + item
            for question, item in zip(questions, retrieved_contexts_by_bigquery)
        ],
        "response": generated_answers_by_gemini,
    }
)

In [13]:
eval_bigquery

,prompt,response
0,Answer the question: Busco una crema para las ...,"Nombre: Duplo Farline Crema De Manos Anti_Age,..."
1,Answer the question: Necesito un suplemento de...,"Nombre: VITAMINAS D3&K2 60CAPS, Descripción: -..."
2,Answer the question: ¿Tienes algún producto pa...,"Nombre: CURLY METHOD PACK, Descripción: CURLY ..."
3,Answer the question: Quiero una crema hidratan...,"Nombre: CREMA REAFIRMANTE 500 ML, Descripción:..."
4,Answer the question: ¿Hay algún spray nasal pa...,Nombre: Pack Bexident Fresh Breath Colutorio +...


In [16]:
#definimos la respuesta del modelo base como la respuesta del modelo bigquery para las metricas pairwise
eval_rerank['baseline_model_response'] = eval_bigquery['response']

In [17]:
eval_rerank

,prompt,response,baseline_model_response
0,Answer the question: Busco una crema para las ...,"Nombre: CREMA ACEITE ROSA MOSQU 50ML, Descripc...","Nombre: Duplo Farline Crema De Manos Anti_Age,..."
1,Answer the question: Necesito un suplemento de...,"Nombre: PEPTIVIS LIMON 20 SOBRES, Descripción:...","Nombre: VITAMINAS D3&K2 60CAPS, Descripción: -..."
2,Answer the question: ¿Tienes algún producto pa...,Nombre: SOLARAY PIEL CABELLO Y UÑAS 60 CAPSULA...,"Nombre: CURLY METHOD PACK, Descripción: CURLY ..."
3,Answer the question: Quiero una crema hidratan...,"Nombre: SCHUSSLER AC FAC AVENA 25ML, Descripci...","Nombre: CREMA REAFIRMANTE 500 ML, Descripción:..."
4,Answer the question: ¿Hay algún spray nasal pa...,Nombre: Duplo Farline Desodorante Spray Sensib...,Nombre: Pack Bexident Fresh Breath Colutorio +...


In [18]:
#definimos la respuesta del modelo base como la respuesta del modelo rerank para las metricas pairwise
eval_gemini['baseline_model_response'] = eval_rerank['response']

In [19]:
eval_gemini

,prompt,response,baseline_model_response
0,Answer the question: Busco una crema para las ...,"{'type': 'product_search', 'message': '¡Hola! ...","Nombre: CREMA ACEITE ROSA MOSQU 50ML, Descripc..."
1,Answer the question: Necesito un suplemento de...,"{'type': 'product_search', 'message': '¡Hola! ...","Nombre: PEPTIVIS LIMON 20 SOBRES, Descripción:..."
2,Answer the question: ¿Tienes algún producto pa...,"{'type': 'product_search', 'message': '¡Hola! ...",Nombre: SOLARAY PIEL CABELLO Y UÑAS 60 CAPSULA...
3,Answer the question: Quiero una crema hidratan...,"{'type': 'product_search', 'message': '¡Hola! ...","Nombre: SCHUSSLER AC FAC AVENA 25ML, Descripci..."
4,Answer the question: ¿Hay algún spray nasal pa...,"{'type': 'product_search', 'message': '¡Hola! ...",Nombre: Duplo Farline Desodorante Spray Sensib...


In [7]:
# ----------------------Metricas----------------------

"""Select and create metrics
You can run evaluation for just one metric, or a combination of metrics.
For this example, we select a few RAG-related predefined metrics, and create a few of our own custom metrics."""

# Explore predefined metrics: https://cloud.google.com/vertex-ai/generative-ai/docs/models/metrics-templates
# See all the available metric examples

MetricPromptTemplateExamples.list_example_metric_names()

['coherence',
 'fluency',
 'safety',
 'groundedness',
 'instruction_following',
 'verbosity',
 'text_quality',
 'summarization_quality',
 'question_answering_quality',
 'multi_turn_chat_quality',
 'multi_turn_safety',
 'pairwise_coherence',
 'pairwise_fluency',
 'pairwise_safety',
 'pairwise_groundedness',
 'pairwise_instruction_following',
 'pairwise_verbosity',
 'pairwise_text_quality',
 'pairwise_summarization_quality',
 'pairwise_question_answering_quality',
 'pairwise_multi_turn_chat_quality',
 'pairwise_multi_turn_safety']

In [ ]:
# See the prompt example for one of the pointwise metrics
print(MetricPromptTemplateExamples.get_prompt_template("question_answering_quality"))

In [22]:
#Run evaluation with your dataset

rag_eval_bigquery = EvalTask(
    dataset=eval_bigquery,
    metrics=[
        "coherence",
        "groundedness",
        "safety",
        "question_answering_quality",
    ],
    experiment=EXPERIMENT,
)

rag_eval_rerank = EvalTask(
    dataset=eval_rerank,
    metrics=[
        "coherence",
        "fluency",
        "groundedness",
        "safety",
        "question_answering_quality",
        "pairwise_coherence",
        "pairwise_fluency",
        "pairwise_safety",
        "pairwise_question_answering_quality"
    ],
    experiment=EXPERIMENT,
)

rag_eval_gemini = EvalTask(
    dataset=eval_gemini,
    metrics=[
        "coherence",
        "fluency",
        "groundedness",
        "safety",
        "question_answering_quality",
        "pairwise_coherence",
        "pairwise_fluency",
        "pairwise_safety",
        "pairwise_question_answering_quality"
    ],
    experiment=EXPERIMENT,
)

In [23]:

result_rag_bigquery = rag_eval_bigquery.evaluate()
result_rag_rerank = rag_eval_rerank.evaluate()
result_rag_gemini = rag_eval_gemini.evaluate()

Associating projects/793914295237/locations/us-central1/metadataStores/default/contexts/rag-eval-01-26310e17-d631-4f43-a858-41ac0d0050a4 to Experiment: rag-eval-01


Computing metrics with a total of 20 Vertex Gen AI Evaluation Service API requests.


100%|██████████| 20/20 [00:22<00:00,  1.10s/it]

All 20 metric requests are successfully computed.
Evaluation Took:22.049837958000353 seconds


Associating projects/793914295237/locations/us-central1/metadataStores/default/contexts/rag-eval-01-473a6c75-8921-4231-a200-ba3136bc655c to Experiment: rag-eval-01


...

All 45 metric requests are successfully computed.
Evaluation Took:46.708055875000355 seconds


Associating projects/793914295237/locations/us-central1/metadataStores/default/contexts/rag-eval-01-2624e249-7787-42b7-856c-9ddebe239b36 to Experiment: rag-eval-01


Pairwise metric `pairwise_coherence` loaded from `MetricPromptTemplateExamples` does not have `baseline_model` specified and only supports Bring-Your-Own-Response(BYOR) evaluation. If you would like to run inference on the baseline model, please instantiate a `PairwiseMetric` and provide the `baseline_model` parameter.
Pairwise metric `pairwise_fluency` loaded from `MetricPromptTemplateExamples` does not have `baseline_model` specified and only supports Bring-Your-Own-Response(BYOR) evaluation. If you would like to run inference on the baseline model, please instantiate a `PairwiseMetric` and provide the `baseline_model` parameter.
Pairwise metric `pairwise_safety` loaded from `MetricPromptTemplateExamples` does not have `baseline_model` specified and only supports Bring-Your-Own-Response(BYOR) evaluation. If you would like to run inference on the baseline model, please instantiate a `PairwiseMetric` and provide the `baseline_model` parameter.
Pairwise metric `pairwise_question_answeri

100%|██████████| 45/45 [00:46<00:00,  1.04s/it]

All 45 metric requests are successfully computed.
Evaluation Took:46.887877667000794 seconds


In [25]:
#------------- Display evaluation results -------------
"""View summary results
If you want to have an overall view of all the metrics from individual model's evaluation
result in one table, you can use the display_eval_report() helper function."""

display_eval_report(
    (
        "Model Bigquery Eval Result",
        result_rag_bigquery.summary_metrics,
        result_rag_bigquery.metrics_table,
    )
)

## Model Bigquery Eval Result

### Summary Metrics

,row_count,coherence/mean,coherence/std,groundedness/mean,groundedness/std,safety/mean,safety/std,question_answering_quality/mean,question_answering_quality/std
0,5.0,1.2,0.447214,1.0,0.0,1.0,0.0,1.4,0.547723


### Report Metrics

,prompt,response,coherence/explanation,coherence/score,groundedness/explanation,groundedness/score,safety/explanation,safety/score,question_answering_quality/explanation,question_answering_quality/score
0,Answer the question: Busco una crema para las ...,"Nombre: Duplo Farline Crema De Manos Anti_Age,...",The prompt is in Spanish and asks for a cream ...,1.0,The response returns relevant information from...,1.0,The AI model's response lists several skin cre...,1.0,The question asks for a cream for stretch mark...,1.0
1,Answer the question: Necesito un suplemento de...,"Nombre: VITAMINAS D3&K2 60CAPS, Descripción: -...",The response is a list of products with their ...,1.0,The response returns the relevant options from...,1.0,All of the content from the AI's response is s...,1.0,The prompt asks for a vitamin D supplement for...,2.0
2,Answer the question: ¿Tienes algún producto pa...,"Nombre: CURLY METHOD PACK, Descripción: CURLY ...","The prompt asks a question in Spanish ""¿Tienes...",2.0,"The question is ""Do you have any hair loss pro...",1.0,All of the products described in the response ...,1.0,The prompt asks for a product for hair loss. T...,2.0
3,Answer the question: Quiero una crema hidratan...,"Nombre: CREMA REAFIRMANTE 500 ML, Descripción:...",The prompt is in Spanish and asks for a moistu...,1.0,The response returns a product from the prompt...,1.0,All the products provided are standard skincar...,1.0,The prompt asks for a moisturizer for sensitiv...,1.0
4,Answer the question: ¿Hay algún spray nasal pa...,Nombre: Pack Bexident Fresh Breath Colutorio +...,"The prompt asks a question, and the response c...",1.0,The response provided by the model is fully gr...,1.0,The AI returns a list of products. None of the...,1.0,The prompt asks for a nasal spray for allergie...,1.0


In [24]:
#------------- Display evaluation results -------------
"""View summary results
If you want to have an overall view of all the metrics from individual model's evaluation
result in one table, you can use the display_eval_report() helper function."""

display_eval_report(
    (
        "Model A Eval Result",
        result_rag_rerank.summary_metrics,
        result_rag_rerank.metrics_table,
    )
)

## Model A Eval Result

### Summary Metrics

,row_count,coherence/mean,coherence/std,fluency/mean,fluency/std,groundedness/mean,groundedness/std,safety/mean,safety/std,question_answering_quality/mean,question_answering_quality/std,pairwise_coherence/candidate_model_win_rate,pairwise_coherence/baseline_model_win_rate,pairwise_fluency/candidate_model_win_rate,pairwise_fluency/baseline_model_win_rate,pairwise_safety/candidate_model_win_rate,pairwise_safety/baseline_model_win_rate,pairwise_question_answering_quality/candidate_model_win_rate,pairwise_question_answering_quality/baseline_model_win_rate
0,5.0,2.2,0.83666,2.8,1.095445,1.0,0.0,1.0,0.0,2.2,0.83666,0.4,0.6,0.0,0.6,0.0,0.0,0.8,0.2


### Report Metrics

,prompt,response,baseline_model_response,coherence/explanation,coherence/score,fluency/explanation,fluency/score,groundedness/explanation,groundedness/score,safety/explanation,...,question_answering_quality/explanation,question_answering_quality/score,pairwise_coherence/explanation,pairwise_coherence/pairwise_choice,pairwise_fluency/explanation,pairwise_fluency/pairwise_choice,pairwise_safety/explanation,pairwise_safety/pairwise_choice,pairwise_question_answering_quality/explanation,pairwise_question_answering_quality/pairwise_choice
0,Answer the question: Busco una crema para las ...,"Nombre: CREMA ACEITE ROSA MOSQU 50ML, Descripc...","Nombre: Duplo Farline Crema De Manos Anti_Age,...",The AI model's response is difficult to follow...,2.0,The AI-generated response exhibits mostly flue...,4.0,The AI has referenced information from the pro...,1.0,All of the suggested products are safe and app...,...,The prompt asks for a stretch mark cream. The ...,2.0,Response A is slightly better because it follo...,BASELINE,Response A is better than Response B based on ...,BASELINE,"Both responses are safe and appropriate, provi...",TIE,Response B is better than Response A because i...,CANDIDATE
1,Answer the question: Necesito un suplemento de...,"Nombre: PEPTIVIS LIMON 20 SOBRES, Descripción:...","Nombre: VITAMINAS D3&K2 60CAPS, Descripción: -...",The AI has provided a somewhat incoherent resp...,3.0,The AI has responded with supplements for olde...,4.0,The response includes information only from th...,1.0,All of the supplements presented are valid rec...,...,The prompt asks for a Vitamin D supplement for...,3.0,Response A provides more options that answer t...,BASELINE,Both responses provide similar product options...,BASELINE,Both responses seem to be hallucinated because...,TIE,Response B is better because it shows some lev...,CANDIDATE
2,Answer the question: ¿Tienes algún producto pa...,Nombre: SOLARAY PIEL CABELLO Y UÑAS 60 CAPSULA...,"Nombre: CURLY METHOD PACK, Descripción: CURLY ...",The AI has extracted several products related ...,2.0,"The AI response, while providing somewhat rele...",2.0,The response provided by the AI model is compl...,1.0,All the products mentioned are hair products t...,...,The prompt asks for products for hair loss. Th...,2.0,Both responses seem to return a similar group ...,CANDIDATE,Both responses are identical in terms of fluen...,TIE,"Both responses are safe and appropriate, addre...",TIE,Both responses provided relevant products rela...,CANDIDATE
3,Answer the question: Quiero una crema hidratan...,"Nombre: SCHUSSLER AC FAC AVENA 25ML, Descripci...","Nombre: CREMA REAFIRMANTE 500 ML, Descripción:...",The AI suggests five products. The prompt asks...,3.0,The AI suggests five products. Out of the five...,2.0,The AI model extracted relevant product inform...,1.0,All of the products provided are safe and appr...,...,The prompt states the user wants a moisturizer...,3.0,Response B omitted relevant information from t...,BASELINE,Both responses give a list of products related...,BASELINE,Both responses provide helpful information in ...,TIE,Both responses include moisturizing creams for...,BASELINE
4,Answer the question: ¿Hay algún spray nasal pa...,Nombre: Duplo Farline Desodorante Spray Sensib...,Nombre: Pack Bexident Fresh Breath Colutorio +...,"The AI model fails to answer the question ""Is ...",1.0,"The AI response, while providing information f...",2.0,The response provided by the model is fully gr...,1.0,The AI response recommends different types of ...,...,The prompt asks a question regarding an allerg...,1.0,"Both responses give a list of products, but re...",CANDIDATE,Both responses had similar issues with formatt...,TIE,Both responses contain the same number of help...,TIE,Both responses presented the context. Response...,CANDIDATE


In [26]:
#------------- Display evaluation results -------------
"""View summary results
If you want to have an overall view of all the metrics from individual model's evaluation
result in one table, you can use the display_eval_report() helper function."""

display_eval_report(
    (
        "Model Gemini Eval Result",
        result_rag_gemini.summary_metrics,
        result_rag_gemini.metrics_table,
    )
)

## Model Gemini Eval Result

### Summary Metrics

,row_count,coherence/mean,coherence/std,fluency/mean,fluency/std,groundedness/mean,groundedness/std,safety/mean,safety/std,question_answering_quality/mean,question_answering_quality/std,pairwise_coherence/candidate_model_win_rate,pairwise_coherence/baseline_model_win_rate,pairwise_fluency/candidate_model_win_rate,pairwise_fluency/baseline_model_win_rate,pairwise_safety/candidate_model_win_rate,pairwise_safety/baseline_model_win_rate,pairwise_question_answering_quality/candidate_model_win_rate,pairwise_question_answering_quality/baseline_model_win_rate
0,5.0,3.8,0.447214,3.8,0.447214,0.8,0.447214,1.0,0.0,2.6,0.894427,0.4,0.6,0.6,0.4,0.8,0.0,0.6,0.4


### Report Metrics

,prompt,response,baseline_model_response,coherence/explanation,coherence/score,fluency/explanation,fluency/score,groundedness/explanation,groundedness/score,safety/explanation,...,question_answering_quality/explanation,question_answering_quality/score,pairwise_coherence/explanation,pairwise_coherence/pairwise_choice,pairwise_fluency/explanation,pairwise_fluency/pairwise_choice,pairwise_safety/explanation,pairwise_safety/pairwise_choice,pairwise_question_answering_quality/explanation,pairwise_question_answering_quality/pairwise_choice
0,Answer the question: Busco una crema para las ...,"{'type': 'product_search', 'message': '¡Hola! ...","Nombre: CREMA ACEITE ROSA MOSQU 50ML, Descripc...",STEP 1: The purpose of the response is to prov...,4.0,The response is mostly fluent (some minor repe...,4.0,The AI response is grounded because the provid...,1.0,"The AI response recommends several products, p...",...,The prompt asks for a cream for stretch marks....,2.0,Response A is slightly better in terms of cohe...,BASELINE,Response A is a list of products with their de...,CANDIDATE,Response A provides correctly formatted produc...,CANDIDATE,Response A is better as Response B is a JSON w...,BASELINE
1,Answer the question: Necesito un suplemento de...,"{'type': 'product_search', 'message': '¡Hola! ...","Nombre: PEPTIVIS LIMON 20 SOBRES, Descripción:...",STEP 1: The purpose is to provide product reco...,4.0,The AI response demonstrates mostly fluent lan...,4.0,The response mentions products exclusively fro...,1.0,The AI's response is safe and provides helpful...,...,The response mentions products containing vita...,3.0,Response B is better than Response A because i...,CANDIDATE,Response B is better overall because it's form...,CANDIDATE,Response B is a safer response as it is format...,CANDIDATE,Response B provided the product information in...,CANDIDATE
2,Answer the question: ¿Tienes algún producto pa...,"{'type': 'product_search', 'message': '¡Hola! ...",Nombre: SOLARAY PIEL CABELLO Y UÑAS 60 CAPSULA...,STEP 1: The purpose of the response is to prov...,4.0,The response exhibits mostly fluent language w...,4.0,"The response mentions two products, SOLARAY PI...",0.0,The response is safe and helpful to the user. ...,...,The response does not follow the instructions ...,2.0,Response A is slightly more coherent than Resp...,BASELINE,Response B is better than Response A because i...,CANDIDATE,Response B is better than Response A because i...,CANDIDATE,Response A provides a list of products that co...,CANDIDATE
3,Answer the question: Quiero una crema hidratan...,"{'type': 'product_search', 'message': '¡Hola! ...","Nombre: SCHUSSLER AC FAC AVENA 25ML, Descripci...",STEP 1: The purpose of the prompt is to search...,4.0,The response has some minor issues with accura...,4.0,The response mentions products and their prope...,1.0,The response recommends products for sensitive...,...,The response recommends products suitable for ...,4.0,Response A is better than Response B because i...,BASELINE,Response A provides a list of products with de...,BASELINE,Response B is safer as it warns against using ...,CANDIDATE,Response A is better since it's more concise a...,BASELINE
4,Answer the question: ¿Hay algún spray nasal pa...,"{'type': 'product_search', 'message': '¡Hola! ...",Nombre: Duplo Farline Desodorante Spray Sensib...,STEP 1: The purpose of the user input is to in...,3.0,The AI response has some grammatical errors su...,3.0,"The response mentions products like ""Bach Resc...",1.0,The AI model's response is safe and does not c...,...,The response does not directly answer the ques...,2.0,Response B is better since it is more relevant...,CANDIDATE,Response A is better than Response B because i...,BASELINE,Both responses are safe in that there is no id...,TIE,Response B is better than Response A because i...,CANDIDATE


In [29]:
#------------- Visualize evaluation results -------------
eval_results = []
eval_results.append(
    ("Model Rerank", result_rag_rerank.summary_metrics, result_rag_rerank.metrics_table)
)    
eval_results.append(
    ("Model Gemini", result_rag_gemini.summary_metrics, result_rag_gemini.metrics_table)
)    
eval_results.append(
    ("Retrieve Bigquery", result_rag_bigquery.summary_metrics, result_rag_bigquery.metrics_table)
)  

plot_radar_plot(
    eval_results,
    metrics=[
        f"{metric}/mean"
        # Edit your list of metrics here if you used other metrics in evaluation.
        for metric in [
            "coherence",
            "groundedness",
            "question_answering_quality",
            "pairwise_coherence",
            "pairwise_fluency",
            "pairwise_safety",
            "pairwise_question_answering_quality"
        ]
    ],
)    

plot_bar_plot(
    eval_results,
    metrics=[
        f"{metric}/mean"
        for metric in [
            "coherence",
            "groundedness",
            "question_answering_quality",
            "pairwise_coherence",
            "pairwise_fluency",
            "pairwise_safety",
            "pairwise_question_answering_quality"
        ]
    ],
)